# Nanodisk Monte Carlo on the generated kernel — ready to run

Upload this file to Kaggle or Colab, or run it on your own machine with an NVIDIA GPU. **It does not
need the repository**: it carries the package modules it uses, each checked against its `sha256`, so
it runs code byte-identical to `spin-colour-kernel` at commit `f2e9bb5`.

## What it does

It cools the magnetic nanodisk (sc Heisenberg disc with exchange up to four shells,
interfacial DMI, cubic and surface anisotropy, Zeeman field) from a hot temperature to 0.1 K, one
state per parameter row, and reports these physical metrics at every
step: E per site, |M|, Mz, Cv, χ and the Berg–Lüscher topological charge Q of the middle layer.

## How to use it

**Edit only the parameters cell.** One row per state; the notebook counts the rows. Then *Run all*.

## Performance

Measured on the paper's benchmarks: on the nanodisk the generated kernel's throughput rises from
5.0 to 13.4 spin flips/ns between 8 and 128 replicas on an H100 (E6), and every state is checked
spin by spin against the sequential chain before it runs.

> Needs a CUDA GPU with compute capability ≥ 7.0 (T4, A100, H100, …). Triton does not run on a P100.

## 1 · Parameters

The only cell to edit.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  THE PARAMETERS — the only cell to edit
# ══════════════════════════════════════════════════════════════════════════
#
# One row per state. Each row carries the SEVEN dataset parameters, IN THIS ORDER:
#
#   index  name   what it is                                   units
#   ─────  ─────  ───────────────────────────────────────────  ─────────────
#     0    Jex2   exchange, 2nd sc shell (distance √2)          relative to Jex
#     1    Jex3   exchange, 3rd sc shell (distance √3)          relative to Jex
#     2    Jex4   exchange, 4th sc shell (distance 2)           relative to Jex
#     3    Kan1   cubic anisotropy, first order                 meV
#     4    KanS   surface anisotropy                            meV
#     5    Hex    external field (magnitude)                    meV
#     6    KDM    interfacial Dzyaloshinskii–Moriya             meV
#
# First-neighbour exchange is Jex = 1 meV, the energy unit. Temperature is not a parameter: every
# state follows the same ladder, so the transition temperature is measured, not declared.
#
# SHELLS AND COLOURINGS. A non-zero Jex2, Jex3 or Jex4 in ANY row switches that shell on for every
# state (one colouring serves the whole run). The colouring is searched and checked on the disk:
#
#   active shells   colours
#   ─────────────   ───────
#   1, 1+3             2
#   1+2                4
#   1+4                3
#   1+3+4              4
#   1+2+4              7
#   1+2+3              8
#   1+2+3+4            9
#
# A sweep is one launch per colour; fewer colours means more parallel work per launch.

PARAMS = [
    # [ Jex2, Jex3, Jex4,  Kan1  , KanS ,  Hex   ,  KDM   ]
    [  0.0,  0.0,  0.0,  0.2867,  0.0 ,  0.7691,  0.6535],  # uniform
    # [  0.0,  0.0,  0.0,  0.1770,  0.139,  0.4830,  0.8800],  # helical
    # [  0.0,  0.0,  0.0,  0.3500,  0.0 ,  0.2000,  0.9500],  # points
    # [  0.0,  0.0,  0.0,  0.1000,  0.200,  0.9000,  0.3000],  # lines
]
NAMES = ["uniform"]          # one name per row; left short, the rest are numbered

# ══════════════════════════════════════════════════════════════════════════
#  GEOMETRY
# ══════════════════════════════════════════════════════════════════════════
RD     = 18.30   # disk radius in lattice steps; the box is N = 2(⌊Rd⌋+1)+1 wide
LAYERS = 5       # layers in z
ANGLE  = 90.0    # field direction in degrees, in the x–z plane

# ══════════════════════════════════════════════════════════════════════════
#  SCHEDULE
# ══════════════════════════════════════════════════════════════════════════
REPLICAS      = 128     # independent chains PER STATE. More is cheaper per update: no penalty
T_START       = 20.0    # K
T_END         = 0.1     # K
N_TEMPS       = 200     # steps: 200 gives exactly 0.1 K steps
SWEEPS        = 20000   # sweeps per temperature (2/3 thermalize, 1/3 measure)
MEASURE_EVERY = 10      # read the observables every this many sweeps of the measuring third
SEED          = 42

# ══════════════════════════════════════════════════════════════════════════
#  GPUS
# ══════════════════════════════════════════════════════════════════════════
GPUS = "all"            # "all", or a list such as [0] or [0, 1]

## 2 · Environment

In [ ]:
# Environment: the GPU, and the two libraries the kernel needs (PyTorch with CUDA, and Triton).
import subprocess, sys

def _sh(*a):
    try:
        return subprocess.run(a, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""

print("GPU:", _sh("nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
                  "--format=csv,noheader") or "none detected")
import torch
if not torch.cuda.is_available():
    raise SystemExit("no CUDA GPU visible: this notebook needs one (Kaggle: Settings > Accelerator)")
try:
    import triton
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton
caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
if any(c < (7, 0) for c in caps):
    raise SystemExit(f"Triton needs compute capability >= 7.0; found {caps}")
print(f"torch {torch.__version__}  triton {triton.__version__}  "
      f"{torch.cuda.device_count()} GPU(s), capability {caps}")

## 3 · The modules

Written to disk from the literals below, each checked against its `sha256`.

In [ ]:
# Spin_Colour_Kernel/__init__.py — byte-identical to the repository. sha256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
import hashlib, os
_src = ''
assert hashlib.sha256(_src.encode()).hexdigest() == "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "__init__.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/__init__.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/__init__.py  sha256 e3b0c44298fc1c14…")

In [ ]:
# Spin_Colour_Kernel/crystals.py — byte-identical to the repository. sha256 13b8e7758dcc9259ee1d7705e6b994857a7a0c80f72e73d6998195ad562dfe76
import hashlib, os
_src = '"""Crystals as Bravais lattices with a basis, and their bond tables (proposal r07, eq. 4).\n\nA site is the pair (c, alpha): an integer cell coordinate c in Z^D and a basis index alpha.\nA directed bond (alpha, alpha2, delta) joins (c, alpha) to (c + delta, alpha2). Bond tables are\ngenerated from neighbour shells by real-space distance, so a crystal is fully described by its\nprimitive (or conventional) vectors and its basis offsets.\n"""\n\nfrom __future__ import annotations\n\nimport itertools\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Bond:\n    """One directed bond-table entry: (c, alpha) ~ (c + delta, alpha2), on neighbour shell `shell`."""\n\n    alpha: int\n    alpha2: int\n    delta: tuple[int, ...]\n    shell: int\n\n\n@dataclass\nclass Crystal:\n    name: str\n    vectors: np.ndarray          # (D, D) rows are lattice vectors a_k in Cartesian units\n    basis: np.ndarray            # (b, D) basis offsets in fractional (cell) coordinates\n    shells: tuple[int, ...] = (1,)\n    bonds: list[Bond] = field(default_factory=list)\n\n    @property\n    def D(self) -> int:\n        return int(self.vectors.shape[0])\n\n    @property\n    def b(self) -> int:\n        return int(self.basis.shape[0])\n\n    def position(self, c, alpha) -> np.ndarray:\n        frac = np.asarray(c, dtype=float) + self.basis[alpha]\n        return frac @ self.vectors\n\n\ndef shell_distances(vectors: np.ndarray, basis: np.ndarray, nshell: int, reach: int = 3) -> list[float]:\n    """The first `nshell` distinct neighbour distances over every basis pair."""\n    D = vectors.shape[0]\n    ds = set()\n    for a in range(basis.shape[0]):\n        for a2 in range(basis.shape[0]):\n            for delta in itertools.product(range(-reach, reach + 1), repeat=D):\n                r = (np.asarray(delta, float) + basis[a2] - basis[a]) @ vectors\n                d = float(np.linalg.norm(r))\n                if d > 1e-9:\n                    ds.add(round(d, 9))\n    return sorted(ds)[:nshell]\n\n\ndef build_bonds(vectors: np.ndarray, basis: np.ndarray, shells: tuple[int, ...], reach: int = 3) -> list[Bond]:\n    """Directed bond table for the requested shells (1-based), in a fixed, documented order.\n\n    Order: shell, then alpha, then alpha2, then delta in lexicographic order. The order is the\n    accumulation order of the local field, both in the kernel and in every reference.\n    """\n    dist = shell_distances(vectors, basis, max(shells), reach)\n    D = vectors.shape[0]\n    out: list[Bond] = []\n    for s in sorted(shells):\n        target = dist[s - 1]\n        for a in range(basis.shape[0]):\n            for a2 in range(basis.shape[0]):\n                for delta in itertools.product(range(-reach, reach + 1), repeat=D):\n                    r = (np.asarray(delta, float) + basis[a2] - basis[a]) @ vectors\n                    if abs(np.linalg.norm(r) - target) < 1e-7:\n                        out.append(Bond(a, a2, tuple(int(v) for v in delta), s))\n    return out\n\n\ndef make(name: str, shells: tuple[int, ...] = (1,)) -> Crystal:\n    """The crystals of the system matrix. bcc and fcc use conventional cubic cells (b = 2, 4)."""\n    s3 = np.sqrt(3.0)\n    tri = np.array([[1.0, 0.0], [0.5, s3 / 2]])\n    table = {\n        "square": (np.eye(2), np.zeros((1, 2))),\n        "triangular": (tri, np.zeros((1, 2))),\n        "honeycomb": (tri, np.array([[0.0, 0.0], [1 / 3, 1 / 3]])),\n        "kagome": (tri, np.array([[0.0, 0.0], [0.5, 0.0], [0.0, 0.5]])),\n        "sc": (np.eye(3), np.zeros((1, 3))),\n        "bcc": (np.eye(3), np.array([[0, 0, 0], [0.5, 0.5, 0.5]])),\n        "fcc": (np.eye(3), np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]])),\n    }\n    if name not in table:\n        raise KeyError(f"unknown crystal {name!r}; known: {sorted(table)}")\n    vec, basis = table[name]\n    vec = np.asarray(vec, float)\n    basis = np.asarray(basis, float)\n    reach = 3 if max(shells) <= 2 else 4\n    return Crystal(name, vec, basis, tuple(shells), build_bonds(vec, basis, tuple(shells), reach))\n\n\ndef coordination(crystal: Crystal) -> dict[int, int]:\n    """Bonds leaving each basis site, per alpha."""\n    out = {a: 0 for a in range(crystal.b)}\n    for bd in crystal.bonds:\n        out[bd.alpha] += 1\n    return out\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "13b8e7758dcc9259ee1d7705e6b994857a7a0c80f72e73d6998195ad562dfe76", "crystals.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/crystals.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/crystals.py  sha256 13b8e7758dcc9259…")

In [ ]:
# Spin_Colour_Kernel/colouring.py — byte-identical to the repository. sha256 86c87e437035d5af8e4ead9416924b598beb80241722d45742de20fa66d26a01
import hashlib, os
_src = '"""Linear colourings as lattice homomorphisms into a finite abelian group (proposal r07, eqs. 5-8).\n\ngamma(c, alpha) = K c + o_alpha (mod m), with K in Z^{r x D} and o_alpha in Z_m1 x ... x Z_mr.\nAdmissibility (7) is a finite check over the bond table; the seam condition (8) binds the colouring\nto a periodic box. The search runs over every abelian group (invariant-factor form) in increasing\norder n_c and returns the first admissible colouring. The clique number of the bond graph is the\nlower bound, so optimality is certified when the two meet.\n"""\n\nfrom __future__ import annotations\n\nimport itertools\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .crystals import Crystal\n\n\n@dataclass(frozen=True)\nclass Colouring:\n    m: tuple[int, ...]                 # group orders, invariant-factor form\n    K: tuple[tuple[int, ...], ...]     # r rows, D columns\n    o: tuple[tuple[int, ...], ...]     # b rows, r columns\n\n    @property\n    def n_colours(self) -> int:\n        return int(np.prod(self.m)) if self.m else 1\n\n    def colour(self, c, alpha) -> int:\n        """Mixed-radix index of gamma(c, alpha) in [0, n_c)."""\n        idx = 0\n        for r, mr in enumerate(self.m):\n            g = (sum(self.K[r][k] * int(c[k]) for k in range(len(c))) + self.o[alpha][r]) % mr\n            idx = idx * mr + g\n        return idx\n\n    def describe(self) -> str:\n        grp = " x ".join(f"Z{m}" for m in self.m) or "trivial"\n        return f"{grp}; K={[list(r) for r in self.K]}; o={[list(x) for x in self.o]}"\n\n\ndef abelian_groups(n: int) -> list[tuple[int, ...]]:\n    """Every abelian group of order n, as invariant factors m1 | m2 | ... | mr."""\n    out = []\n\n    def rec(rem, prev, acc):\n        if rem == 1:\n            out.append(tuple(acc))\n            return\n        for d in range(2, rem + 1):\n            if rem % d == 0 and (prev is None or d % prev == 0):\n                rec(rem // d, d, acc + [d])\n\n    if n == 1:\n        return [()]\n    rec(n, None, [])\n    # invariant factors must divide each other in order; rec builds m1 | m2 | ...\n    return [g for g in out if all(g[i + 1] % g[i] == 0 for i in range(len(g) - 1))]\n\n\ndef admissible(col: Colouring, crystal: Crystal) -> bool:\n    """Condition (7): K delta + o_alpha2 - o_alpha != 0 (mod m) for every bond."""\n    for bd in crystal.bonds:\n        v = []\n        for r, mr in enumerate(col.m):\n            v.append((sum(col.K[r][k] * bd.delta[k] for k in range(crystal.D))\n                      + col.o[bd.alpha2][r] - col.o[bd.alpha][r]) % mr)\n        if all(x == 0 for x in v):\n            return False\n    return True\n\n\ndef seam_ok(col: Colouring, box: tuple[int, ...], periodic: tuple[bool, ...]) -> bool:\n    """Condition (8): every periodic axis k has K[:, k] L_k = 0 (mod m)."""\n    for k, (Lk, per) in enumerate(zip(box, periodic)):\n        if per and any((col.K[r][k] * Lk) % mr for r, mr in enumerate(col.m)):\n            return False\n    return True\n\n\ndef search(crystal: Crystal, box: tuple[int, ...] | None = None, periodic: tuple[bool, ...] | None = None,\n           max_colours: int = 32) -> Colouring | None:\n    """First admissible linear colouring in increasing n_c (and seam-compatible with `box` if given)."""\n    D, b = crystal.D, crystal.b\n    for n in range(1, max_colours + 1):\n        for m in abelian_groups(n):\n            r = len(m)\n            elems = list(itertools.product(*[range(mi) for mi in m])) if r else [()]\n            # K columns: one group element per lattice axis\n            for cols in itertools.product(elems, repeat=D):\n                K = tuple(tuple(cols[k][i] for k in range(D)) for i in range(r))\n                for offs in itertools.product(elems, repeat=b - 1):\n                    o = (tuple([0] * r),) + tuple(offs)\n                    col = Colouring(m, K, o)\n                    if box is not None and not seam_ok(col, box, periodic or (True,) * D):\n                        continue\n                    if admissible(col, crystal):\n                        return col\n    return None\n\n\ndef verify_on_box(col: Colouring, crystal: Crystal, box: tuple[int, ...], periodic: tuple[bool, ...]) -> bool:\n    """Independent check: colour every site of the box and test every bond explicitly, seam included."""\n    for c in itertools.product(*[range(L) for L in box]):\n        for bd in crystal.bonds:\n            if bd.alpha >= crystal.b:\n                continue\n            c2 = []\n            inside = True\n            for k in range(crystal.D):\n                x = c[k] + bd.delta[k]\n                if periodic[k]:\n                    x %= box[k]\n                elif not 0 <= x < box[k]:\n                    inside = False\n                c2.append(x)\n            if inside and col.colour(c, bd.alpha) == col.colour(c2, bd.alpha2):\n                return False\n    return True\n\n\ndef clique_number(crystal: Crystal, window: int = 2) -> int:\n    """Largest clique of the infinite bond graph, by Bron-Kerbosch on a window around one site.\n\n    Every clique is translated so that one of its sites is (0, alpha); every other site lies within\n    one bond of it, so a window of the bond reach holds every clique.\n    """\n    D = crystal.D\n    reach = max(max(abs(x) for x in bd.delta) for bd in crystal.bonds)\n    w = max(window, reach)\n    nodes = [(c, a) for c in itertools.product(range(-w, w + 1), repeat=D) for a in range(crystal.b)]\n    index = {n: i for i, n in enumerate(nodes)}\n\n    nbrs = [set() for _ in nodes]\n    for i, u in enumerate(nodes):\n        for bd in crystal.bonds:\n            if bd.alpha != u[1]:\n                continue\n            c2 = tuple(u[0][k] + bd.delta[k] for k in range(D))\n            j = index.get((c2, bd.alpha2))\n            if j is not None:\n                nbrs[i].add(j)\n    best = 1\n    for a in range(crystal.b):\n        start = index[(tuple([0] * D), a)]\n\n        def bk(R, P, X):\n            nonlocal best\n            if not P and not X:\n                best = max(best, len(R))\n                return\n            if len(R) + len(P) <= best:\n                return\n            for v in list(P):\n                bk(R | {v}, P & nbrs[v], X & nbrs[v])\n                P = P - {v}\n                X = X | {v}\n\n        bk({start}, set(nbrs[start]), set())\n    return best\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "86c87e437035d5af8e4ead9416924b598beb80241722d45742de20fa66d26a01", "colouring.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/colouring.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/colouring.py  sha256 86c87e437035d5af…")

In [ ]:
# Spin_Colour_Kernel/system.py — byte-identical to the repository. sha256 f3755bd076a9c0cd28a3da7dc7c774d2db5188f5e84085dce981f276c33fde03
import hashlib, os
_src = '"""A system is the triple (spin model, crystal, finite geometry) with its couplings (proposal r07, eqs. 1-4).\n\nSpins live in the component-planar, basis-planar layout of eq. (10):\n    S[B, n, b, L3, L2, L1], fastest axis L1 (x). For D = 2, L3 = 1.\nMasked sites hold the zero vector and are never updated, so a masked or open-boundary neighbour\ncontributes zero to the local field (3) without a mask load.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom . import crystals as cr\nfrom .colouring import Colouring, search, seam_ok, verify_on_box\n\n\n@dataclass\nclass System:\n    name: str\n    n: int                                   # spin components: 1 Ising, 2 XY, 3 Heisenberg\n    crystal: cr.Crystal\n    box: tuple[int, ...]                     # (L1, L2[, L3]) cells\n    periodic: tuple[bool, ...]\n    J: dict[int, float]                      # isotropic exchange per shell\n    extra: dict[int, np.ndarray] = field(default_factory=dict)   # bond index -> extra n x n matrix (DMI ...)\n    h: tuple[float, ...] | None = None       # uniform field\n    cubic_k1: np.ndarray | None = None       # per-site cubic anisotropy K1, shape (b, L3, L2, L1)\n    cubic_k2: float = 0.0                    # (sx sy sz)^2 coefficient\n    mask: np.ndarray | None = None           # bool (b, L3, L2, L1); False = no spin\n    colouring: Colouring | None = None\n    T_ref: float | None = None               # reference critical temperature, when one exists\n    acceptance: str = "metropolis"           # "metropolis": u < exp(-beta dE); "barker": heat-bath rule\n\n    def __post_init__(self):\n        if self.n not in (1, 2, 3):\n            raise ValueError("n must be 1, 2 or 3")\n        if len(self.box) != self.crystal.D or len(self.periodic) != self.crystal.D:\n            raise ValueError("box and periodic must have one entry per lattice axis")\n        if self.cubic_k1 is not None and self.n != 3:\n            raise ValueError("cubic anisotropy needs n = 3")\n        if self.acceptance not in ("metropolis", "barker"):\n            raise ValueError("acceptance must be \'metropolis\' or \'barker\'")\n        if self.colouring is None:\n            self.colouring = search(self.crystal, self.box, self.periodic)\n            if self.colouring is None:\n                raise ValueError(f"{self.name}: no seam-compatible linear colouring for box {self.box}")\n        if not seam_ok(self.colouring, self.box, self.periodic):\n            raise ValueError(f"{self.name}: colouring violates the seam condition (8) on box {self.box}")\n        if not verify_on_box(self.colouring, self.crystal, self.box, self.periodic):\n            raise ValueError(f"{self.name}: colouring is not admissible on the box")\n\n    # ---------------------------------------------------------------- geometry\n    @property\n    def L1(self) -> int:\n        return int(self.box[0])\n\n    @property\n    def L2(self) -> int:\n        return int(self.box[1])\n\n    @property\n    def L3(self) -> int:\n        return int(self.box[2]) if self.crystal.D == 3 else 1\n\n    @property\n    def b(self) -> int:\n        return self.crystal.b\n\n    @property\n    def shape(self) -> tuple[int, int, int, int]:\n        return (self.b, self.L3, self.L2, self.L1)\n\n    def site_mask(self) -> np.ndarray:\n        return np.ones(self.shape, bool) if self.mask is None else self.mask.astype(bool)\n\n    @property\n    def n_active(self) -> int:\n        return int(self.site_mask().sum())\n\n    def bond_matrix(self, i: int) -> np.ndarray:\n        bd = self.crystal.bonds[i]\n        M = self.J.get(bd.shell, 0.0) * np.eye(self.n)\n        if i in self.extra:\n            M = M + self.extra[i]\n        return M\n\n    def colour_map(self) -> np.ndarray:\n        """Colour of every site, shape (b, L3, L2, L1)."""\n        out = np.zeros(self.shape, np.int64)\n        for a in range(self.b):\n            for z in range(self.L3):\n                for y in range(self.L2):\n                    for x in range(self.L1):\n                        c = (x, y, z)[: self.crystal.D]\n                        out[a, z, y, x] = self.colouring.colour(c, a)\n        return out\n\n\n# -------------------------------------------------------------------- matrix\ndef ising(crystal: str, L: int, J: float = 1.0, shells=(1,), T_ref=None) -> System:\n    c = cr.make(crystal, tuple(shells))\n    return System(f"Ising {crystal}", 1, c, (L,) * c.D, (True,) * c.D, {s: J for s in shells}, T_ref=T_ref)\n\n\ndef xy(crystal: str, L: int, J: float = 1.0, T_ref=None) -> System:\n    c = cr.make(crystal)\n    return System(f"XY {crystal}", 2, c, (L,) * c.D, (True,) * c.D, {1: J}, T_ref=T_ref)\n\n\ndef heisenberg(crystal: str, L: int, J: float = 1.0, T_ref=None, label=None) -> System:\n    c = cr.make(crystal)\n    return System(label or f"Heisenberg {crystal}", 3, c, (L,) * c.D, (True,) * c.D, {1: J}, T_ref=T_ref)\n\n\ndef nanodot(Rd: float = 18.30, L: int = 5, Jex=1.0, Kan1=0.177, Kan2=0.034, KanS=0.139, Hex=0.483,\n            KDM=0.88, gamma_deg=90.0, shells=(1,), J_shells=None) -> System:\n    """The nanodot system: sc Heisenberg disc inside an open box.\n\n    `shells` adds sc exchange shells beyond the first, with couplings `J_shells` {shell: J} relative\n    to Jex; the colouring search then finds a colouring admissible for all of them. DMI stays on the\n    first-shell in-plane bonds and the surface is still defined by first neighbours.\n\n    The disc never reaches the box edge, so every seam bond would touch a masked (zero) site; an\n    open box is the same Hamiltonian as a periodic one and meets condition (8) without a constraint on N.\n\n    Interfacial DMI on in-plane bonds (the local field term\n    KDM * (-dyz, dxz, dyx - dxy)), cubic anisotropy with surface constant KanS where a site has\n    fewer than six in-disc neighbours, and a Zeeman field Hex (cos g, 0, sin g).\n    """\n    shells = tuple(sorted(set(shells) | {1}))\n    c = cr.make("sc", shells)\n    J = {1: Jex, **{int(k): float(v) for k, v in (J_shells or {}).items() if int(k) in shells and int(k) != 1}}\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    xs = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(xs, xs, indexing="xy")\n    disk = (xg ** 2 + yg ** 2) <= Rd ** 2               # [y, x]\n    mask = np.broadcast_to(disk, (L, N, N)).copy()[None]  # (1, L, N, N)\n    surface = np.zeros_like(mask)\n    for z in range(L):\n        for y in range(N):\n            for x in range(N):\n                if not mask[0, z, y, x]:\n                    continue\n                k = 0\n                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0), (0, -1, 0), (1, 0, 0), (-1, 0, 0)):\n                    nz, ny, nx = z + dz, (y + dy) % N, (x + dx) % N\n                    if 0 <= nz < L and mask[0, nz, ny, nx]:\n                        k += 1\n                surface[0, z, y, x] = k < 6\n    k1 = np.where(surface, KanS, Kan1).astype(np.float32) * mask\n    extra = {}\n    for i, bd in enumerate(c.bonds):\n        dx, dy, dz = bd.delta\n        E = np.zeros((3, 3))\n        if (dx, dy, dz) == (1, 0, 0):\n            E[1, 2], E[2, 1] = KDM, -KDM\n        elif (dx, dy, dz) == (-1, 0, 0):\n            E[1, 2], E[2, 1] = -KDM, KDM\n        elif (dx, dy, dz) == (0, 1, 0):\n            E[0, 2], E[2, 0] = -KDM, KDM\n        elif (dx, dy, dz) == (0, -1, 0):\n            E[0, 2], E[2, 0] = KDM, -KDM\n        if E.any():\n            extra[i] = E\n    g = np.deg2rad(gamma_deg)\n    return System("Heisenberg nanodot", 3, c, (N, N, L), (False, False, False), J, extra,\n                  h=(Hex * float(np.cos(g)), 0.0, Hex * float(np.sin(g))), cubic_k1=k1, cubic_k2=Kan2,\n                  mask=mask)\n\n\n# Reference critical temperatures (J = kB = 1), from experiments v03 section E4.\nTC = {\n    "Ising square": 2.0 / np.log(1 + np.sqrt(2)),\n    "Ising triangular": 4.0 / np.log(3.0),\n    "Ising sc": 4.511523,      # 1/0.22165463, Ferrenberg, Xu and Landau 2018\n    "Ising bcc": 6.355,        # 1/0.157373, Lundow and Campbell 2018\n    "Ising fcc": 9.797132,     # 1/0.1020707(2), Yu 2015 (Physica A 419, 75)\n    "XY sc": 2.20184,          # 1/0.4541652, Xu et al. 2019\n    "Heisenberg sc": 1.4430,   # Chen, Ferrenberg and Landau 1993\n    "Heisenberg bcc": 2.0542,  # Chen, Ferrenberg and Landau 1993\n}\n\n\ndef matrix(small: bool = True) -> list[System]:\n    """The 13 systems of experiment E1 at a small size per crystal (seam-compatible boxes)."""\n    s = [\n        ising("square", 8, T_ref=TC["Ising square"]),\n        ising("triangular", 6, T_ref=TC["Ising triangular"]),\n        ising("sc", 4, T_ref=TC["Ising sc"]),\n        ising("bcc", 4, T_ref=TC["Ising bcc"]),\n        ising("fcc", 4, T_ref=TC["Ising fcc"]),\n        xy("square", 8, T_ref=0.893),\n        xy("sc", 4, T_ref=TC["XY sc"]),\n        heisenberg("sc", 4, T_ref=TC["Heisenberg sc"]),\n        heisenberg("bcc", 4, T_ref=TC["Heisenberg bcc"]),\n        heisenberg("fcc", 4, T_ref=None),\n        heisenberg("triangular", 6, J=-1.0, T_ref=0.285, label="Heisenberg triangular AFM"),\n        heisenberg("kagome", 6, J=-1.0, T_ref=None, label="Heisenberg kagome AFM"),\n        nanodot(Rd=3.0, L=3),\n    ]\n    return s\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "f3755bd076a9c0cd28a3da7dc7c774d2db5188f5e84085dce981f276c33fde03", "system.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/system.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/system.py  sha256 f3755bd076a9c0cd…")

In [ ]:
# Spin_Colour_Kernel/generator.py — byte-identical to the repository. sha256 5e9058a6ce269efcd6b61030bc90d836d2853b18bf683f7ae06e0da598241694
import hashlib, os
_src = '"""The kernel generator (proposal r07, section 5): one System -> one specialized Triton kernel.\n\nSpins are read in the storage type of `src_ptr` (int8 for Ising, float32 for vector models) and\nconverted to float32 for the arithmetic; results are stored back in the type of `dst_ptr`.\n\nThe model (n, proposal, anisotropy), the bond table, the colouring and the box are emitted as\nstraight-line Triton source; only the replica index, the inverse temperature and the counter state\nare runtime values. One launch updates one (colour, sublattice) class; a sweep is the composition\nof the n_c * b class launches, in a fixed order.\n\nColour compaction. Along x a linear colouring repeats with period P = lcm_r m_r / gcd(K_r1, m_r)\n(STRIDE = P), and along a row each colour of the row\'s coset appears at most once per period. A class\nlaunch therefore gives each program a tile of R rows x BX = ceil(L1 / P) sites: each lane solves its\nrow\'s residue x0 from the colouring and takes x0 + P * j. A row without the colour does nothing, and\nwhen P is close to L1 one program still covers R rows instead of a single site. Every load of a class launch is masked by the update predicate, so a sweep reads the\nlattice about once whatever n_c is, instead of n_c times. The sequential reference and the synchronous control keep STRIDE = 1, R = 1.\n\nRandomness is Philox (`tl.rand4x`) keyed by a per-sweep seed and countered by (replica, site), so the\nstream is a pure function of the site and the sweep, never of the launch grid. The same compiled\nkernel serves the parallel chain and the sequential reference: the reference launches it once per\nsite (`x_only`, `row0`), the parallel chain once per class.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport math\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .system import System\n\nCACHE = Path(os.environ.get("SCK_CACHE", Path.home() / ".cache" / "spin_colour_kernel"))\n\n\ndef _f(x: float) -> str:\n    return repr(float(np.float32(x)))\n\n\ndef _coord_expr(var: str, d: int, L: int, periodic: bool) -> tuple[str, str | None]:\n    """Neighbour coordinate and its validity predicate (None when always valid)."""\n    if d == 0:\n        return var, None\n    if periodic:\n        return f"(({var} + {d % L}) % {L})", None\n    e = f"({var} + {d})"\n    return e, f"(({e} >= 0) & ({e} < {L}))"\n\n\ndef x_period(col) -> int:\n    """Period along x of a linear colouring: lcm over its factors of m_r / gcd(K_r1, m_r)."""\n    P = 1\n    for r, mr in enumerate(col.m):\n        P = math.lcm(P, mr // math.gcd(int(col.K[r][0]) % mr or mr, mr))\n    return P\n\n\ndef _colour_expr(sys_: System, xv: str, a: int) -> str:\n    """Colour of site (xv, y, z) of sublattice a, as a Triton expression (mixed-radix, eq. 6)."""\n    col, D = sys_.colouring, sys_.crystal.D\n    if not col.m:\n        return f"{xv} * 0"\n    terms = []\n    for r, mr in enumerate(col.m):\n        coeffs = list(col.K[r]) + [0] * (3 - D)\n        terms.append((f"({coeffs[0]} * {xv} + {coeffs[1]} * y + {coeffs[2]} * z + {col.o[a][r]}) % {mr}", mr))\n    expr = terms[0][0]\n    for e, mr in terms[1:]:\n        expr = f"({expr}) * {mr} + {e}"\n    return expr\n\n\ndef _emit_tile(w, sys_: System, chunked: bool) -> None:\n    """Lanes, rows and the colour-compacted x coordinate of one class launch; defines live, xc, yv, zv.\n\n    chunked: one-row programs take their x chunk from grid axis 2 (rows longer than one program\'s lanes)."""\n    b, L1, L2, L3 = sys_.b, sys_.L1, sys_.L2, sys_.L3\n    # a program covers a tile of R rows x BX sites, flattened onto BLOCK lanes: lane l is row l // BX\n    # of the tile and the (l % BX)-th site of that row\'s colour class (x0 + STRIDE * j)\n    w("    lane = tl.arange(0, BLOCK)")\n    w("    if R == 1:")                       # one row per program: row coordinates stay scalars\n    w("        j = lane + tl.program_id(2) * BLOCK" if chunked else "        j = lane")\n    w("        row = tl.program_id(1) + row0")\n    w(f"        z = row // {L2}")\n    w(f"        y = row % {L2}")\n    w("    else:")\n    w("        j = lane % BX")\n    w("        rowv = tl.program_id(1) * R + lane // BX + row0")\n    w(f"        rok = (lane < R * BX) & (rowv < {L3 * L2})")\n    w("        rowc = tl.where(rok, rowv, 0)")\n    w(f"        z = rowc // {L2}")\n    w(f"        y = rowc % {L2}")\n\n    # compaction: along a row the colour has period STRIDE and each colour of the row\'s coset appears\n    # at most once per period, so each lane finds its row\'s residue x0 (L1 when the row lacks it)\n    w("    if STRIDE > 1:")\n    w(f"        x0 = tl.zeros([BLOCK], dtype=tl.int32) + {L1}")\n    w("        for q in tl.static_range(STRIDE):")\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"            {kw} ALPHA == {a}:")\n        w(f"                cq = {_colour_expr(sys_, \'q\', a)}")\n        w("                x0 = tl.where(cq == COLOUR, q, x0)")\n    w("    else:")\n    w("        x0 = 0")\n    w("    x = x0 + STRIDE * j")\n    w("    xo = tl.zeros([BLOCK], dtype=tl.int32) + x_only")\n    w("    if R == 1:")\n    w(f"        live = (x < {L1}) & ((x == xo) | (xo < 0))")\n    w("    else:")\n    w(f"        live = rok & (x < {L1}) & ((x == xo) | (xo < 0))")\n    w("    xc = tl.where(live, x, 0)")\n    w("    yv = tl.zeros([BLOCK], dtype=tl.int32) + y")\n    w("    zv = tl.zeros([BLOCK], dtype=tl.int32) + z")\n\n\ndef _emit_class(w, sys_: System) -> None:\n    """Update predicate of one (colour, sublattice) class: defines colour, upd, own, m_own (masked systems), ld."""\n    b, L1, L2 = sys_.b, sys_.L1, sys_.L2\n    pl = sys_.L3 * L2 * L1\n    has_mask = sys_.mask is not None\n    # colour mask, from (6), per alpha\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        w(f"        colour = {_colour_expr(sys_, \'xc\', a)}")\n    w("    if COLOUR < 0:")\n    w("        upd = live")\n    w("    else:")\n    w("        upd = live & (colour == COLOUR)")\n    w(f"    own = ALPHA * {pl} + z * {L2 * L1} + y * {L1} + xc")\n    if has_mask:\n        w("    m_own = tl.load(mask_ptr + own, mask=live, other=0)")\n        w("    upd = upd & (m_own != 0)")\n    # a class launch reads only the sites it updates; the synchronous control reads every live site\n    w("    if SYNC:")\n    w("        ld = live")\n    w("    else:")\n    w("        ld = upd")\n\n\ndef source(sys_: System, packed: bool = False, bitwise: bool = False, independent: bool = False) -> str:\n    """Triton source text for one system.\n\n    bitwise (Ising only): true multi-spin coding, see `bitwise.source`; same packed storage, but the\n    update stays in bit operations with one uniform per (word, site, sweep) and a host acceptance table.\n    independent (bitwise only): one 32-bit uniform per replica instead, compared bit-sliced.\n\n    packed (Ising only): multi-spin coding over replicas. Word w of a site holds that site\'s spin in\n    replicas 32 w .. 32 w + 31, one bit each (1 = +1). A program loads one word per neighbour, unpacks it\n    into a [BLOCK, 32] tile, runs the same Metropolis step with the same Philox counter per (replica,\n    site) as the unpacked kernel, and packs the result back with an OR. Rows longer than BLOCK are split\n    over grid axis 2."""\n    if independent and not bitwise:\n        raise ValueError("independent per-replica draws need the bitwise kernel")\n    if bitwise:\n        from .bitwise import source as bitwise_source\n        return bitwise_source(sys_, independent)\n    n, b = sys_.n, sys_.b\n    if packed and n != 1:\n        raise ValueError("bit packing is for Ising (n = 1) only")\n    P = packed\n    RB = "BLOCK, 32" if P else "BLOCK"\n    L1, L2, L3 = sys_.L1, sys_.L2, sys_.L3\n    D = sys_.crystal.D\n    per = list(sys_.periodic) + [False] * (3 - D)\n    pl = L3 * L2 * L1                      # one (component, alpha) plane\n    nsite = b * pl\n    has_mask = sys_.mask is not None\n    has_cubic = sys_.cubic_k1 is not None\n    lines: list[str] = []\n    w = lines.append\n    w("import triton")\n    w("import triton.language as tl")\n    w("")\n    w("")\n    if P:\n        w("@triton.jit")\n        w("def _or(a, b):")\n        w("    return a | b")\n        w("")\n        w("")\n    w(\'@triton.jit(do_not_specialize=["x_only", "row0"])\')\n    w("def kernel(src_ptr, dst_ptr, mask_ptr, k1_ptr, beta_ptr, state_ptr, x_only, row0,")\n    w("           ALPHA: tl.constexpr, COLOUR: tl.constexpr, BLOCK: tl.constexpr, SYNC: tl.constexpr,")\n    w("           STRIDE: tl.constexpr, BX: tl.constexpr, R: tl.constexpr):")\n    if P:\n        w("    wrd = tl.program_id(0)")\n        w("    rr = tl.arange(0, 32)")\n        w("    rep = wrd * 32 + rr")\n    else:\n        w("    rep = tl.program_id(0)")\n    _emit_tile(w, sys_, True)     # one-row programs may be chunked over grid axis 2 (packed: PTILE; float: MAX_BLOCK)\n    w("    beta = tl.load(beta_ptr + rep)")\n    w("    base_seed = tl.load(state_ptr + 0)")\n    w("    sweep = tl.load(state_ptr + 1)")\n    w("    seed = ((base_seed * 1000003 + sweep * 7919 + 12345) % 2147483629).to(tl.int32)")\n    w(f"    rbase = wrd.to(tl.int64) * {nsite}" if P else f"    rbase = rep.to(tl.int64) * {n * nsite}")\n    _emit_class(w, sys_)\n    # own spin\n    if P:\n        w("    w0 = tl.load(src_ptr + rbase + own, mask=ld, other=0)")\n        w("    s0 = (((w0[:, None] >> rr[None, :]) & 1) * 2 - 1).to(tl.float32)")\n    else:\n        for k in range(n):\n            w(f"    s{k} = tl.load(src_ptr + rbase + {k * nsite} + own, mask=ld, other=0).to(tl.float32)")\n    # local field, per alpha\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        groups: dict[float, list[int]] = {}\n        extras: list[int] = []\n        for i, bd in enumerate(sys_.crystal.bonds):\n            if bd.alpha != a:\n                continue\n            groups.setdefault(float(sys_.J.get(bd.shell, 0.0)), []).append(i)\n            if i in sys_.extra:\n                extras.append(i)\n        loaded = {}\n        for k in range(n):\n            w(f"        v{k} = tl.zeros([{RB}], dtype=tl.float32)")\n        idx = 0\n        for Jv, members in groups.items():\n            if Jv == 0.0 and not any(i in sys_.extra for i in members):\n                continue\n            for k in range(n):\n                w(f"        g{k} = tl.zeros([{RB}], dtype=tl.float32)")\n            for i in members:\n                bd = sys_.crystal.bonds[i]\n                d = list(bd.delta) + [0] * (3 - D)\n                ex, vx = _coord_expr("xc", d[0], L1, per[0])\n                ey, vy = _coord_expr("yv", d[1], L2, per[1])\n                ez, vz = _coord_expr("zv", d[2], L3, per[2])\n                valid = " & ".join(["ld"] + [p for p in (vx, vy, vz) if p])\n                w(f"        q = {bd.alpha2 * pl} + {ez} * {L2 * L1} + {ey} * {L1} + {ex}")\n                w(f"        ok = {valid}")\n                w("        q = tl.where(ok, q, 0)")\n                if P and has_mask:      # a masked neighbour holds no spin: it contributes 0, as in the unpacked store\n                    w("        ok = ok & (tl.load(mask_ptr + q, mask=ok, other=0) != 0)")\n                for k in range(n):\n                    if P:\n                        w("        tw = tl.load(src_ptr + rbase + q, mask=ok, other=0)")\n                        w("        t = tl.where(ok[:, None], (((tw[:, None] >> rr[None, :]) & 1) * 2 - 1).to(tl.float32), 0.0)")\n                    else:\n                        w(f"        t = tl.load(src_ptr + rbase + {k * nsite} + q, mask=ok, other=0).to(tl.float32)")\n                    w(f"        g{k} = g{k} + t")\n                    if i in sys_.extra:\n                        w(f"        n{idx}_{k} = t")\n                if i in sys_.extra:\n                    loaded[i] = idx\n                    idx += 1\n            if Jv != 0.0:\n                for k in range(n):\n                    w(f"        v{k} = v{k} + {_f(Jv)} * g{k}")\n        for i in extras:\n            E = sys_.extra[i]\n            j = loaded[i]\n            for k in range(n):\n                terms = [f"{_f(E[k, l])} * n{j}_{l}" for l in range(n) if E[k, l] != 0.0]\n                if terms:\n                    w(f"        v{k} = v{k} + ({\' + \'.join(terms)})")\n    if sys_.h is not None:\n        for k in range(n):\n            if sys_.h[k] != 0.0:\n                w(f"    v{k} = v{k} + {_f(sys_.h[k])}")\n    # proposal (9)\n    w(f"    off = (rep[None, :] * {nsite} + own[:, None]).to(tl.int32)" if P else f"    off = (rep * {nsite} + own).to(tl.int32)")\n    w("    u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    if n == 1:\n        w("    t0 = -s0")\n    elif n == 2:\n        w("    phi = 6.283185307179586 * u1")\n        w("    t0 = tl.cos(phi)")\n        w("    t1 = tl.sin(phi)")\n    else:\n        w("    cz = 2.0 * u1 - 1.0")\n        w("    phi = 6.283185307179586 * u2")\n        w("    rr = tl.sqrt(tl.maximum(1.0 - cz * cz, 0.0))")\n        w("    t0 = rr * tl.cos(phi)")\n        w("    t1 = rr * tl.sin(phi)")\n        w("    t2 = cz")\n    # energy change (3)\n    dE = " + ".join(f"(t{k} - s{k}) * v{k}" for k in range(n))\n    w(f"    dE = -({dE})")\n    if has_cubic:\n        w("    k1 = tl.load(k1_ptr + own, mask=ld, other=0.0)")\n        k2 = _f(sys_.cubic_k2)\n        for p in ("t", "s"):\n            w(f"    a_{p} = k1 * (({p}0 * {p}1) * ({p}0 * {p}1) + ({p}0 * {p}2) * ({p}0 * {p}2) "\n              f"+ ({p}1 * {p}2) * ({p}1 * {p}2)) + {k2} * ({p}0 * {p}1 * {p}2) * ({p}0 * {p}1 * {p}2)")\n        w("    dE = dE + (a_t - a_s)")\n    bt, ud = ("beta[None, :]", "upd[:, None]") if P else ("beta", "upd")\n    if sys_.acceptance == "barker":\n        # heat-bath (Barker) rule: accept with 1 / (1 + exp(beta dE)); dE = 0 is accepted with probability 1/2\n        w(f"    acc = (u0 * (1.0 + tl.exp({bt} * dE)) < 1.0) & {ud}")\n    else:\n        w(f"    acc = (u0 < tl.exp(-{bt} * dE)) & {ud}")\n    if P:\n        w("    up = (tl.where(acc, t0, s0) > 0).to(tl.int32) << rr[None, :]")\n        w("    r0 = tl.reduce(up, 1, _or)")\n    else:\n        for k in range(n):\n            w(f"    r{k} = tl.where(acc, t{k}, s{k}).to(dst_ptr.dtype.element_ty)")\n    w("    if SYNC:")\n    wm = "live & (m_own != 0)" if has_mask else "live"\n    for k in range(n):\n        w(f"        tl.store(dst_ptr + rbase + {k * nsite} + own, r{k}, mask={wm})")\n    w("    else:")\n    for k in range(n):\n        w(f"        tl.store(dst_ptr + rbase + {k * nsite} + own, r{k}, mask=upd)")\n    return "\\n".join(lines) + "\\n"\n\n\ndef compile_system(sys_: System, packed: bool = False, bitwise: bool = False, independent: bool = False):\n    """Write the generated source to the cache, import it, and return the jitted kernel."""\n    src = source(sys_, packed, bitwise, independent)\n    key = hashlib.sha256(src.encode()).hexdigest()[:16]\n    CACHE.mkdir(parents=True, exist_ok=True)\n    path = CACHE / f"sck_{key}.py"\n    if not path.exists():\n        path.write_text(src)\n    name = f"sck_{key}"\n    if name in sys.modules:\n        return sys.modules[name].kernel, src\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod.kernel, src\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "5e9058a6ce269efcd6b61030bc90d836d2853b18bf683f7ae06e0da598241694", "generator.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/generator.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/generator.py  sha256 5e9058a6ce269efc…")

In [ ]:
# Spin_Colour_Kernel/bitwise.py — byte-identical to the repository. sha256 a4488d0ada766a05b1381a32f9744a867849e8547d40e8d9c6c03d36822f225d
import hashlib, os
_src = '"""Bitwise multi-spin-coded Ising: asynchronous multi-spin coding over replicas (routes B and C).\n\nStorage is the packed word layout of `generator.source(packed=True)`: word w of a site holds that site\'s\nspin in replicas 32 w .. 32 w + 31, one bit each (1 = +1). Unlike the packed path, which unpacks every word\nto a [BLOCK, 32] float tile, this kernel never leaves the bits:\n\n* per neighbour j present in the box and the mask, x = s_i XOR s_j marks the anti-aligned replicas, and a\n  ripple adder accumulates x into NP bit planes, so plane p holds bit p of the anti-aligned count k of every\n  replica (NP = bit length of the largest coordination, 4 for fcc);\n* the energy change of flipping s_i is dE = 2 J (z_eff - 2 k) + 2 h s_i, with z_eff the number of present\n  neighbours of the site (the same for all 32 replicas), so the acceptance probability is a host table\n  p[word, s, z_eff, k] built per temperature for the system\'s rule (Metropolis min(1, e^{-beta dE}) or\n  Barker 1 / (1 + e^{beta dE}));\n* one uniform u per (word, site, sweep) is shared by the 32 replicas of the word, and the flip mask is\n  OR_k [(u < p(z_eff, k[, s])) ? eq_k : 0], with eq_k the replicas whose count equals k.\n\nEach replica is still an exact Metropolis or Barker chain; replicas of one word are correlated through\nthe shared uniform (and may coalesce), so independent samples come from different words.\n\nRandomness: Philox keyed by the per-sweep seed of the other kernels. Uniform (word w, site i) is output\nw % 4 of the call with counter (w // 4) * nsite + i, so one call serves four words and a program that\ncovers an aligned group of four words wastes no output. The stream is a function of (word, site, sweep)\nonly, never of the launch grid; `uniforms` reproduces it in NumPy for the sequential reference.\n\nIndependent mode (`source(sys_, independent=True)`, `Sampler(..., bitwise=True, independent=True)`): the\nshared uniform makes the replicas of a word coalesce (at Tc on the square lattice 32 replicas collapse to two\nconfigurations within about 50 sweeps), so this mode gives every replica (bit lane) its own draw and keeps the\nupdate exact and bitwise:\n\n* the table holds P = floor(p 2^32) as an integer in [0, 2^32] (`p_fixed`); a replica flips iff its 32-bit\n  uniform integer U satisfies U < P, an exact Bernoulli(P / 2^32) draw. p is therefore quantized down to a\n  multiple of 2^-32; p = 1 gives P = 2^32 (always, no random bits) and P = 0 never flips;\n* U is compared bit-sliced over the 32 lanes, most significant bit first: at level l a random word R_l holds\n  bit 31 - l of U of every lane and Pb_l the same bit of every lane\'s P (selected by the count planes, and by\n  s when h != 0); accept |= und & ~R_l & Pb_l, und &= ~(R_l ^ Pb_l). Lanes start undecided iff 0 < P < 2^32;\n  each level halves them, so the program exits as soon as its whole tile is decided, and after 32 levels the\n  rest (U == P) is rejected, exactly;\n* R_{4g} .. R_{4g+3} of (word w, site i) are the four raw outputs of `tl.randint4x(key_g, w nsite + i)` with\n  the int32 counter of the shared mode and the per-group key key_g = (seed + (g + 1) GROUP_KEY_STEP) mod\n  2147483629, a function of (word, site, level group, sweep) only; `lane_uniforms` rebuilds every replica\'s U\n  in NumPy. (A 64-bit offset would switch Triton\'s GPU randint4x to 64-bit Philox, which the interpreter does\n  not do, so the level group goes into the key, not into the counter.)\n\nOnly systems with one Ising coupling J on every bond that carries one (any crystal, any shells of equal J),\nan optional uniform field and an optional site mask qualify; `check` refuses the others.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .system import System\n\n#: replicas per word\nWORD = 32\n\n\n# ------------------------------------------------------------------ system\ndef check(sys_: System) -> tuple[float, float, int]:\n    """(J, h, Z) of a system the bitwise kernel can run, Z the largest coordination; ValueError otherwise."""\n    if sys_.n != 1:\n        raise ValueError("bitwise multi-spin coding is for Ising (n = 1) only")\n    if sys_.extra:\n        raise ValueError("bitwise multi-spin coding needs isotropic exchange only (no extra bond matrices)")\n    if sys_.cubic_k1 is not None:\n        raise ValueError("bitwise multi-spin coding does not support anisotropy")\n    Js = {float(sys_.J.get(bd.shell, 0.0)) for bd in sys_.crystal.bonds} - {0.0}\n    if len(Js) != 1:\n        raise ValueError(f"bitwise multi-spin coding needs one uniform coupling J on every bond, got {sorted(Js)}")\n    h = float(sys_.h[0]) if sys_.h is not None else 0.0\n    return Js.pop(), h, max(len(bonds(sys_, a)) for a in range(sys_.b))\n\n\ndef bonds(sys_: System, a: int) -> list:\n    """Bonds leaving sublattice a that carry a coupling, in bond-table order."""\n    return [bd for bd in sys_.crystal.bonds if bd.alpha == a and sys_.J.get(bd.shell, 0.0) != 0.0]\n\n\ndef p_table(sys_: System, beta: np.ndarray) -> np.ndarray:\n    """Acceptance probabilities p[word, s, z_eff, k], float32, for one inverse temperature per word.\n\n    s = 0 for spin -1 and 1 for +1, z_eff the number of present neighbours, k <= z_eff the anti-aligned\n    ones; entries with k > z_eff are never read and hold 0."""\n    return _p64(sys_, beta).astype(np.float32)\n\n\ndef p_fixed(sys_: System, beta: np.ndarray) -> np.ndarray:\n    """Independent mode: P = floor(p 2^32) in [0, 2^32], int64, same layout as `p_table` (p in float64).\n\n    A replica with 32-bit uniform integer U flips iff U < P: probability P / 2^32, i.e. p rounded down to a\n    multiple of 2^-32 (P = 2^32 when p = 1: always; P = 0: never)."""\n    P = np.floor(_p64(sys_, beta) * 2.0 ** 32).astype(np.int64)\n    return np.clip(P, 0, 2 ** 32)\n\n\ndef _p64(sys_: System, beta: np.ndarray) -> np.ndarray:\n    J, h, Z = check(sys_)\n    beta = np.asarray(beta, np.float64).reshape(-1)\n    p = np.zeros((beta.size, 2, Z + 1, Z + 1), np.float64)\n    for s in (0, 1):\n        sigma = 2 * s - 1\n        for ze in range(Z + 1):\n            k = np.arange(ze + 1)\n            dE = 2.0 * J * (ze - 2 * k) + 2.0 * h * sigma\n            x = beta[:, None] * dE[None, :]\n            if sys_.acceptance == "barker":\n                p[:, s, ze, : ze + 1] = 0.5 * (1.0 - np.tanh(0.5 * x))\n            else:\n                p[:, s, ze, : ze + 1] = np.exp(-np.maximum(x, 0.0))\n    return p\n\n\n# ------------------------------------------------------------------ randomness\n_M = 0xFFFFFFFF\n\n\ndef philox_uniform4(seed: int, counters) -> np.ndarray:\n    """NumPy twin of Triton\'s `tl.rand4x(seed, counters)` for an int32 seed and int32 counters: (4, N) float32."""\n    out = philox_raw4(seed, counters).view(np.int32)\n    out = np.where(out < 0, ~out, out)\n    return out.astype(np.float32) * np.float32(4.6566127342e-10)\n\n\ndef philox_raw4(seed: int, c0, c1=0, rounds: int = 10) -> np.ndarray:\n    """NumPy twin of Triton\'s `tl.randint4x(seed, off)` with off = c1 2^32 + c0 (c0 < 2^32): (4, N) uint32.\n\n    Philox-4x32-10 with key (seed low, seed high) and counter (c0, c1, 0, 0)."""\n    c0 = np.asarray(c0, np.int64).reshape(-1)\n    c1 = np.broadcast_to(np.asarray(c1, np.int64).reshape(-1), c0.shape)\n    c0, c1 = (c0 & _M).astype(np.uint64), (c1 & _M).astype(np.uint64)\n    c2 = np.zeros_like(c0)\n    c3 = np.zeros_like(c0)\n    k0 = np.uint64(int(seed) & _M)\n    k1 = np.uint64((int(seed) >> 32) & _M)\n    A, B = np.uint64(0xD2511F53), np.uint64(0xCD9E8D57)\n    m, s32 = np.uint64(_M), np.uint64(32)\n    for _ in range(rounds):\n        p0, p2 = B * c2, A * c0           # < 2^64: exact in uint64\n        c0, c1, c2, c3 = ((p0 >> s32) ^ c1 ^ k0) & m, p0 & m, ((p2 >> s32) ^ c3 ^ k1) & m, p2 & m\n        k0 = (k0 + np.uint64(0x9E3779B9)) & m\n        k1 = (k1 + np.uint64(0xBB67AE85)) & m\n    return np.stack([c0, c1, c2, c3]).astype(np.uint32)\n\n\ndef sweep_seed(base_seed: int, sweep: int) -> int:\n    """The per-sweep Philox key of every generated kernel."""\n    return ((int(base_seed) * 1000003 + int(sweep) * 7919 + 12345) % 2147483629)\n\n\ndef uniforms(base_seed: int, sweep: int, n_words: int, nsite: int) -> np.ndarray:\n    """The kernel\'s uniform of every (word, site) in one sweep: shape (n_words, nsite), float32."""\n    groups = -(-n_words // 4)\n    ctr = np.arange(groups, dtype=np.int64)[:, None] * nsite + np.arange(nsite, dtype=np.int64)[None, :]\n    u = philox_uniform4(sweep_seed(base_seed, sweep), ctr.reshape(-1)).reshape(4, groups, nsite)\n    return u.transpose(1, 0, 2).reshape(groups * 4, nsite)[:n_words]\n\n\n#: independent mode: random words per (word, site, sweep), one per bit of U, four per Philox call\nLEVELS = 32\n#: independent mode: the Philox key of level group g is (seed + (g + 1) GROUP_KEY_STEP) mod 2147483629\nGROUP_KEY_STEP = 104729\n#: independent mode: Philox-4x32 rounds per call, Triton\'s default 10 (the kernel and the NumPy twin agree); 7 rounds\n#: is about 15% faster and is kept as an ablation (experiments v11/v12), not as the default\nINDEPENDENT_ROUNDS = 10\n\n\ndef group_key(seed: int, g: int) -> int:\n    return (int(seed) + (int(g) + 1) * GROUP_KEY_STEP) % 2147483629\n\n\ndef lane_uniforms(base_seed: int, sweep: int, n_words: int, nsite: int) -> np.ndarray:\n    """Independent mode: the 32-bit uniform integer U of every (replica, site) in one sweep, (32 n_words, nsite)\n    int64. Bit 31 - l of U of replica 32 w + j is bit j of the random word R_l(w, site), R_{4g + q} being\n    output q of the Philox call with key `group_key(seed, g)` and counter w nsite + site."""\n    ctr = (np.arange(n_words, dtype=np.int64)[:, None] * nsite + np.arange(nsite, dtype=np.int64)[None, :]).reshape(-1)\n    seed = sweep_seed(base_seed, sweep)\n    lanes = np.arange(WORD, dtype=np.uint64)\n    U = np.zeros((n_words, WORD, nsite), np.int64)\n    for g in range(LEVELS // 4):\n        R = philox_raw4(group_key(seed, g), ctr, rounds=INDEPENDENT_ROUNDS).astype(np.uint64).reshape(4, n_words, 1, nsite)\n        for q in range(4):\n            bit = ((R[q] >> lanes[None, :, None]) & np.uint64(1)).astype(np.int64)     # (n_words, 32, nsite)\n            U |= bit << (LEVELS - 1 - (4 * g + q))\n    return U.reshape(n_words * WORD, nsite)\n\n\n# ------------------------------------------------------------------ sequential reference\ndef classes(sys_: System) -> list[tuple[int, int]]:\n    """(colour, sublattice) classes in sweep order: colour-major, then sublattice; empty classes skipped."""\n    cm, m = sys_.colour_map(), sys_.site_mask()\n    return [(c, a) for c in range(sys_.colouring.n_colours) for a in range(sys_.b) if np.any((cm[a] == c) & m[a])]\n\n\ndef reference_sweep(sys_: System, spins: np.ndarray, ptab: np.ndarray, base_seed: int, sweep: int,\n                    independent: bool = False) -> np.ndarray:\n    """One sequential sweep of every replica on unpacked spins, no bit operations.\n\n    spins: (B, b, L3, L2, L1) of +-1 (0 on masked sites). For each class in sweep order and each of its sites\n    in raster order, every replica r computes its local sum sigma_i sum_j s_j over the present neighbours,\n    k = (z_eff - that sum) / 2, draws the uniform of (word r // 32, site, sweep) and flips when\n    u < ptab[r // 32, s, z_eff, k]. independent: ptab is the integer table of `p_fixed` and replica r flips iff\n    its own integer U (`lane_uniforms`) satisfies U < ptab[r // 32, s, z_eff, k]. Returns the updated copy."""\n    _, h, _ = check(sys_)\n    S = np.array(spins, dtype=np.int64, copy=True)\n    B = S.shape[0]\n    b, L3, L2, L1 = sys_.shape\n    D = sys_.crystal.D\n    box = (L1, L2, L3)\n    per = list(sys_.periodic) + [False] * (3 - D)\n    m = sys_.site_mask()\n    cm = sys_.colour_map()\n    nsite = b * L3 * L2 * L1\n    W = -(-B // WORD)\n    if independent:\n        UL = lane_uniforms(base_seed, sweep, W, nsite)\n    else:\n        U = uniforms(base_seed, sweep, W, nsite)\n    word = np.arange(B) // WORD\n    plane = 1 if h != 0.0 else None\n    for c, a in classes(sys_):\n        for z, y, x in np.argwhere((cm[a] == c) & m[a]):\n            nb = []\n            for bd in bonds(sys_, a):\n                d = list(bd.delta) + [0] * (3 - D)\n                q = [x + d[0], y + d[1], z + d[2]]\n                ok = True\n                for k in range(3):\n                    if per[k]:\n                        q[k] %= box[k]\n                    elif not 0 <= q[k] < box[k]:\n                        ok = False\n                if ok and m[bd.alpha2, q[2], q[1], q[0]]:\n                    nb.append((bd.alpha2, q[2], q[1], q[0]))\n            ze = len(nb)\n            si = S[:, a, z, y, x]\n            tot = sum((S[:, a2, z2, y2, x2] for a2, z2, y2, x2 in nb), np.zeros(B, np.int64))\n            kk = (ze - si * tot) // 2\n            s_idx = (si > 0).astype(np.int64) if plane else np.zeros(B, np.int64)\n            p = ptab[word, s_idx, ze, kk]\n            site = a * L3 * L2 * L1 + z * L2 * L1 + y * L1 + x\n            if independent:\n                u = UL[:B, site]                           # one integer per replica, compared as an integer\n            else:\n                u = U[word, site]\n            S[:, a, z, y, x] = np.where(u < p, -si, si)\n    return S.astype(spins.dtype)\n\n\n# ------------------------------------------------------------------ kernel source\ndef source(sys_: System, independent: bool = False) -> str:\n    """Triton source of the bitwise kernel; same launch structure and class predicate as `generator.source`.\n\n    Grid axis 0 enumerates blocks of G words (G a power of two, a constexpr). One Philox call per site\n    lane and aligned group of four words yields the uniforms of those four words; for G > 4 the G // 4\n    calls of a lane are interleaved back to word order with tl.join.\n\n    independent: one 32-bit uniform per replica, compared bit-sliced with the integer table of `p_fixed`\n    (module docstring); p_ptr then points at that int64 table."""\n    from .generator import _coord_expr, _emit_class, _emit_tile\n\n    J, h, Z = check(sys_)\n    b, L1, L2, L3 = sys_.b, sys_.L1, sys_.L2, sys_.L3\n    D = sys_.crystal.D\n    per = list(sys_.periodic) + [False] * (3 - D)\n    pl = L3 * L2 * L1\n    nsite = b * pl\n    has_mask = sys_.mask is not None\n    NP = max(1, Z.bit_length())\n    Z1 = Z + 1\n    NT = 2 * Z1 * Z1\n    lines: list[str] = []\n    w = lines.append\n    w(\'"""Bitwise multi-spin-coded Ising kernel (generated by Spin_Colour_Kernel.bitwise)."""\')\n    w("import triton")\n    w("import triton.language as tl")\n    w("")\n    w("")\n    w(\'@triton.jit(do_not_specialize=["x_only", "row0", "nword"])\')\n    w("def kernel(src_ptr, dst_ptr, mask_ptr, k1_ptr, p_ptr, state_ptr, x_only, row0, nword,")\n    w("           ALPHA: tl.constexpr, COLOUR: tl.constexpr, BLOCK: tl.constexpr, SYNC: tl.constexpr,")\n    w("           STRIDE: tl.constexpr, BX: tl.constexpr, R: tl.constexpr, G: tl.constexpr):")\n    w("    grp = tl.program_id(0)")\n    w("    cw = tl.arange(0, G)")\n    w("    wv = grp * G + cw")\n    w("    wok = wv < nword")\n    _emit_tile(w, sys_, True)\n    w("    base_seed = tl.load(state_ptr + 0)")\n    w("    sweep = tl.load(state_ptr + 1)")\n    w("    seed = ((base_seed * 1000003 + sweep * 7919 + 12345) % 2147483629).to(tl.int32)")\n    w(f"    rbase = wv.to(tl.int64) * {nsite}")\n    _emit_class(w, sys_)\n    w("    ldw = ld[:, None] & wok[None, :]")\n    w("    s0 = tl.load(src_ptr + rbase[None, :] + own[:, None], mask=ldw, other=0)")\n    w("    ze = tl.zeros([BLOCK], dtype=tl.int32)")\n    for p in range(NP):\n        w(f"    c{p} = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        bl = bonds(sys_, a)\n        if not bl:\n            w("        pass")\n        for bd in bl:\n            d = list(bd.delta) + [0] * (3 - D)\n            ex, vx = _coord_expr("xc", d[0], L1, per[0])\n            ey, vy = _coord_expr("yv", d[1], L2, per[1])\n            ez, vz = _coord_expr("zv", d[2], L3, per[2])\n            valid = " & ".join(["ld"] + [p for p in (vx, vy, vz) if p])\n            w(f"        q = {bd.alpha2 * pl} + {ez} * {L2 * L1} + {ey} * {L1} + {ex}")\n            w(f"        ok = {valid}")\n            w("        q = tl.where(ok, q, 0)")\n            if has_mask:      # a masked neighbour holds no spin: it is not a neighbour at all\n                w("        ok = ok & (tl.load(mask_ptr + q, mask=ok, other=0) != 0)")\n            w("        ze = ze + ok.to(tl.int32)")\n            w("        tw = tl.load(src_ptr + rbase[None, :] + q[:, None], mask=ok[:, None] & wok[None, :], other=0)")\n            w("        cy = tl.where(ok[:, None], s0 ^ tw, 0)")\n            # ripple adder: plane p receives the carry out of plane p - 1\n            for p in range(NP):\n                if p < NP - 1:\n                    w(f"        t = c{p} & cy")\n                    w(f"        c{p} = c{p} ^ cy")\n                    w("        cy = t")\n                else:\n                    w(f"        c{p} = c{p} ^ cy")\n    if independent:\n        _emit_independent(w, sys_, nsite, NP, Z1, NT, h != 0.0)\n    else:\n        _emit_shared(w, nsite, NP, Z1, NT, h != 0.0)\n    w("    r0 = s0 ^ acc")\n    w("    if SYNC:")\n    wm = "live & (m_own != 0)" if has_mask else "live"\n    w(f"        tl.store(dst_ptr + rbase[None, :] + own[:, None], r0, mask=({wm})[:, None] & wok[None, :])")\n    w("    else:")\n    w("        tl.store(dst_ptr + rbase[None, :] + own[:, None], r0, mask=upd[:, None] & wok[None, :])")\n    return "\\n".join(lines) + "\\n"\n\n\ndef _emit_shared(w, nsite: int, NP: int, Z1: int, NT: int, field: bool) -> None:\n    """Flip mask acc from one uniform per (word, site) shared by the word\'s 32 replicas."""\n    # one Philox call per site lane: output w % 4 of counter (w // 4) * nsite + site\n    w("    if G <= 4:")      # the program\'s words lie in one group of four\n    w(f"        off = (((grp * G) // 4) * {nsite} + own).to(tl.int32)")\n    w("        u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    w("        cq = (wv % 4)[None, :]")\n    w("        u = tl.where(cq == 0, u0[:, None], tl.where(cq == 1, u1[:, None], tl.where(cq == 2, u2[:, None], u3[:, None])))")\n    w("    else:")           # G // 4 whole groups: one call per (site, group), outputs interleaved to word order\n    w(f"        gq = grp * (G // 4) + tl.arange(0, G // 4)")\n    w(f"        off = (gq[None, :] * {nsite} + own[:, None]).to(tl.int32)")\n    w("        u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    w("        u = tl.reshape(tl.join(tl.join(u0, u2), tl.join(u1, u3)), [BLOCK, G])")\n    w(f"    tb = p_ptr + wv[None, :] * {NT} + ze[:, None] * {Z1}")\n    w("    acc = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for k in range(Z1):\n        terms = [f"c{p}" if (k >> p) & 1 else f"~c{p}" for p in range(NP)]\n        w(f"    eq = {\' & \'.join(terms)}")\n        if field:\n            w(f"    pm = tl.load(tb + {k}, mask=ldw, other=0.0)")\n            w(f"    pp = tl.load(tb + {Z1 * Z1 + k}, mask=ldw, other=0.0)")\n            w("    acc = acc | tl.where(u < pp, eq & s0, 0) | tl.where(u < pm, eq & ~s0, 0)")\n        else:\n            w(f"    pk = tl.load(tb + {k}, mask=ldw, other=0.0)")\n            w("    acc = acc | tl.where(u < pk, eq, 0)")\n\n\ndef _emit_independent(w, sys_: System, nsite: int, NP: int, Z1: int, NT: int, field: bool) -> None:\n    """Flip mask acc from one 32-bit uniform per replica, compared bit-sliced with P = floor(p 2^32).\n\n    Table entries (one per k, and per s when field) are selected by lane masks e{i}: the lanes whose count\n    is k (and whose spin is s). Lanes with P = 2^32 accept outright, lanes with 0 < P < 2^32 start undecided,\n    and each level of U, most significant first, decides the lanes whose random bit differs from P\'s bit."""\n    w(f"    tb = p_ptr + wv[None, :] * {NT} + ze[:, None] * {Z1}")\n    J, h, Z = check(sys_)\n    # Metropolis on a periodic, unmasked lattice: every site has z_eff = Z, so the sign of dE of each entry is\n    # known here and the entries with dE <= 0 (p = 1, P = 2^32 in the table) accept without a load or a draw\n    static = sys_.acceptance != "barker" and sys_.mask is None and all(sys_.periodic)\n    w("    acc = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    ent = []           # (lane-mask expression, table offset)\n    for k in range(Z1):\n        terms = [f"c{p}" if (k >> p) & 1 else f"~c{p}" for p in range(NP)]\n        w(f"    eq = {\' & \'.join(terms)}")\n        for sel, off, sigma in ((["eq & ~s0", k, -1], ["eq & s0", Z1 * Z1 + k, 1]) if field else (["eq", k, -1],)):\n            if static and k <= Z and 2.0 * J * (Z - 2 * k) + 2.0 * h * sigma <= 0.0:\n                w(f"    acc = acc | ({sel})")\n                continue\n            w(f"    e{len(ent)} = {sel}")\n            ent.append(off)\n    w("    und = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for i, off in enumerate(ent):\n        if static:     # z_eff = Z everywhere: one table value per word, shape [1, G]\n            w(f"    P{i} = tl.load(p_ptr + wv[None, :] * {NT} + {Z * Z1 + off}, mask=wok[None, :], other=0)")\n        else:\n            w(f"    P{i} = tl.load(tb + {off}, mask=ldw, other=0)")\n        w(f"    acc = acc | tl.where(P{i} >= {2 ** 32}, e{i}, 0)")\n        w(f"    und = und | tl.where((P{i} > 0) & (P{i} < {2 ** 32}), e{i}, 0)")\n        # undecided lanes have P < 2^32: the level loop works on 32-bit words\n        w(f"    P{i} = P{i}.to(tl.uint32)")\n    # Philox counter word * nsite + site (int32, as in the shared mode), key per level group: grid-independent\n    w(f"    ctr = (rbase[None, :] + own[:, None].to(tl.int64)).to(tl.int32)")\n\n    def group(indent: str, gexpr: str):\n        w(f"{indent}kg = ((seed.to(tl.int64) + ({gexpr} + 1) * {GROUP_KEY_STEP}) % 2147483629).to(tl.int32)")\n        w(f"{indent}q0, q1, q2, q3 = tl.randint4x(kg, ctr, n_rounds={INDEPENDENT_ROUNDS})")\n        for q in range(4):\n            w(f"{indent}rw = q{q}.to(tl.int32, bitcast=True)")\n            w(f"{indent}sh = ({LEVELS - 1 - q} - 4 * {gexpr}).to(tl.uint32)")\n            # bit sh of each entry\'s P as an all-ones / all-zeros mask, then one AND per entry\n            masks = " | ".join(f"(e{i} & (0 - ((P{i} >> sh) & 1)).to(tl.int32, bitcast=True))" for i in range(len(ent))) or "0"\n            w(f"{indent}pb = {masks}")\n            w(f"{indent}acc = acc | (und & ~rw & pb)")\n            w(f"{indent}und = und & ~(rw ^ pb)")\n\n    # the first two level groups (8 levels) run unconditionally: they decide all but about 2^-8 of the lanes,\n    # and the tile-wide reduction is paid only once, for the rare tail; the stream and the result are unchanged\n    if ent:\n        group("    ", "(tl.program_id(0) * 0).to(tl.int64)")\n        group("    ", "(tl.program_id(0) * 0 + 1).to(tl.int64)")\n    w("    g = tl.program_id(0) * 0 + 2")\n    w("    nund = tl.sum(tl.sum((und != 0).to(tl.int32), axis=1), axis=0)")\n    w(f"    while (g < {LEVELS // 4}) & (nund > 0):")\n    group("        ", "g.to(tl.int64)")\n    w("        g = g + 1")\n    w("        nund = tl.sum(tl.sum((und != 0).to(tl.int32), axis=1), axis=0)")\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "a4488d0ada766a05b1381a32f9744a867849e8547d40e8d9c6c03d36822f225d", "bitwise.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/bitwise.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/bitwise.py  sha256 a4488d0ada766a05…")

In [ ]:
# Spin_Colour_Kernel/sampler.py — byte-identical to the repository. sha256 86a9893be5ac75b74ca1e1a75acf15aadb336e4c5f1514453f6e32977fd3c52b
import hashlib, os
_src = '"""Launch plumbing for a generated kernel: sweeps, the sequential reference, observables.\n\nA parallel sweep is n_c * b launches (colour-major, then sublattice). The sequential reference\nvisits the same classes in the same order and, inside each class, the sites one at a time in\nraster order (z, y, x), through the same compiled kernel. Because no two sites of one class are\nbonded (7), the two chains must hold identical spins after every sweep (hypothesis H1).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\n\nfrom . import bitwise as bw\nfrom .generator import compile_system, x_period\nfrom .system import System\n\n\n#: smallest x-period worth compacting; E6 on a T4 (2026-11-07) lost 25-35% with stride-2 loads\nCOMPACT_MIN = 4\n#: lanes a compacted program aims for: R rows of BX sites each\nTILE = 128\n#: float kernels: lanes per one-row program. A longer row is split over grid axis 2. Above 1024 lanes a 4-warp\n#: Heisenberg program spills its registers to local memory (Nsight on the H100: 0, 18 M and 253 M local sectors at\n#: rows of 1024, 2048 and 4096 sites), so rows up to 1024 sites keep one program, longer rows are chunked.\nMAX_BLOCK = 1024\n#: bit-packed Ising: lanes per program (each lane carries a 32-replica tile, so BLOCK x 32 values)\nPTILE = 64\n#: replicas per packed word\nWORD = 32\n#: bitwise: default words per program (one Philox call per site serves an aligned group of four words)\nBWG = 4\n\n\ndef _num_warps(block: int) -> int:\n    return 4 if block >= 256 else (2 if block >= 128 else 1)\n\n\nclass Sampler:\n    def __init__(self, system: System, batch: int, device="cuda", seed: int = 0, compact: bool = True,\n                 spin_dtype: torch.dtype | None = None, packed: bool | None = None, bitwise: bool = False,\n                 words_per_program: int | None = None, independent: bool = False, bitwise_tile: int | None = None,\n                 bitwise_warps: int | None = None, max_block: int = MAX_BLOCK):\n        """bitwise: true multi-spin coding (module `bitwise`): packed storage updated with bit operations, one\n        uniform per (word, site, sweep) shared by the word\'s 32 replicas, one temperature per word.\n        independent (bitwise only): every replica draws its own 32-bit uniform U and flips iff U < floor(p 2^32),\n        compared bit-sliced; replicas of a word no longer share randomness (p quantized down to 2^-32).\n        words_per_program: bitwise words a program covers (power of two, default BWG; fewer when the batch is small)."""\n        self.sys = system\n        self.bitwise = bool(bitwise)\n        self.independent = bool(independent)\n        if self.independent and not self.bitwise:\n            raise ValueError("independent per-replica draws need bitwise=True")\n        if self.bitwise:\n            _, _, self.zmax = bw.check(system)     # ValueError for systems the bitwise kernel cannot run\n            if packed is False:\n                raise ValueError("bitwise multi-spin coding uses the packed storage")\n            packed = True\n        # Ising defaults to multi-spin coding (32 replicas per int32 word, see generator.source) once the\n        # batch fills a word; below that a word computes 32 replicas for fewer than 32 used, and int8 is faster\n        self.packed = (system.n == 1 and int(batch) >= WORD) if packed is None else bool(packed)\n        if self.packed and system.n != 1:\n            raise ValueError("bit packing is for Ising (n = 1) only")\n        # Ising spins are +-1 (0 on masked sites): int8 storage reads a quarter of the bytes of float32 and holds\n        # them exactly; vector models keep float32 components\n        self.spin_dtype = spin_dtype or (torch.int8 if system.n == 1 else torch.float32)\n        self.B = int(batch)\n        self.device = torch.device(device)\n        self.kernel, self.src = compile_system(system, self.packed, self.bitwise, self.independent)\n        n, b, L3, L2, L1 = system.n, system.b, system.L3, system.L2, system.L1\n        self.W = -(-self.B // WORD) if self.packed else self.B      # program rows of grid axis 0\n        self.Bp = self.W * WORD if self.packed else self.B           # replicas the kernel computes\n        if self.Bp * b * L3 * L2 * L1 >= 2 ** 31 - 1:\n            raise ValueError("replica batch too large for the int32 Philox counter")\n        # bitwise: a program covers G words, a power of two; G >= 4 wastes no Philox output\n        gmax = words_per_program or BWG\n        if gmax & (gmax - 1):\n            raise ValueError("words_per_program must be a power of two")\n        self.G = min(gmax, 1 << (self.W - 1).bit_length()) if self.bitwise else 1\n        self.ngroups = -(-self.W // self.G)\n        self.nrows = L3 * L2\n        if max_block < 16 or max_block & (max_block - 1):\n            raise ValueError("max_block must be a power of two >= 16")\n        self.block = min(max(16, 1 << (L1 - 1).bit_length()), max_block)\n        self.chunks = -(-L1 // self.block)            # one-row programs over grid axis 2 (rows longer than max_block)\n        self.nw = _num_warps(self.block)\n        # colour compaction: a program covers R rows x BX sites of one colour (BX = ceil(L1 / P))\n        P = x_period(system.colouring) if compact else 1\n        P = P if P >= COMPACT_MIN else 1       # below it, strided loads cost more than the masked lanes save\n        self.stride = P\n        self.bx = -(-L1 // P)                         # sites of one colour per row\n        tile_c = bitwise_tile if (bitwise and bitwise_tile) else TILE   # bitwise: lanes per compacted program\n        self.rows_per = max(1, min(self.nrows, tile_c // self.bx)) if P > 1 else 1\n        if self.packed and not self.bitwise:   # a bitwise lane holds G int32 words, not a 32-float tile\n            self.rows_per = max(1, min(self.nrows, PTILE // self.bx)) if P > 1 else 1\n        self.cblock = max(16, 1 << (self.rows_per * self.bx - 1).bit_length())\n        self.cchunks = 1\n        if not self.packed and self.rows_per == 1 and self.cblock > max_block:\n            self.cblock = max_block\n            self.cchunks = -(-self.bx // max_block)\n        self.cnw = _num_warps(self.cblock)\n        if self.packed:   # one-row programs cover at most PTILE sites of the row; longer rows go over grid axis 2\n            nx = self.bx if P > 1 else L1\n            tile = TILE if self.bitwise else PTILE\n            self.pblock = min(tile, max(16, 1 << (nx - 1).bit_length()))\n            self.pchunks = -(-nx // self.pblock)\n            self.pblock_ref = min(tile, self.block)\n            self.pchunks_ref = -(-L1 // self.pblock_ref)\n            if not self.bitwise:\n                self.cnw = 4 if self.cblock * WORD >= 2048 else 2\n            elif bitwise_tile:\n                self.pblock = min(bitwise_tile, max(16, 1 << (nx - 1).bit_length()))\n                self.pchunks = -(-nx // self.pblock)\n            if self.bitwise and bitwise_warps:     # tuning knob of the bitwise kernels (E6 pilot)\n                self.cnw = self.nw = int(bitwise_warps)\n        dev = self.device\n        m = system.site_mask()\n        self.mask_np = m\n        self.mask = torch.as_tensor(m.astype(np.int8), device=dev)\n        k1 = system.cubic_k1 if system.cubic_k1 is not None else np.zeros(m.shape, np.float32)\n        self.k1 = torch.as_tensor(np.asarray(k1, np.float32), device=dev)\n        self.state = torch.tensor([int(seed), 0], dtype=torch.int64, device=dev)\n        self.beta = torch.ones(self.Bp, dtype=torch.float32, device=dev)\n        if self.bitwise:     # acceptance table p[word, s, z_eff, k], rebuilt in place by set_temperature\n            self._table = bw.p_fixed if self.independent else bw.p_table     # int64 P = floor(p 2^32), or float32 p\n            self.ptab = torch.as_tensor(self._table(system, np.ones(self.W)), device=dev)\n        self.colours = system.colour_map()\n        self.classes = [(c, a) for c in range(system.colouring.n_colours) for a in range(b)\n                        if np.any((self.colours[a] == c) & m[a])]\n        mt = torch.as_tensor(m, device=dev)\n        self._mask_f = mt.to(torch.float32)\n\n    # ------------------------------------------------------------ state\n    def set_temperature(self, T):\n        T = torch.as_tensor(T, dtype=torch.float32, device=self.device).reshape(-1)\n        beta = (1.0 / T).expand(self.B).contiguous() if T.numel() == 1 else (1.0 / T).contiguous()\n        if self.Bp > self.B:     # padding replicas of the last word: computed, never observed\n            beta = torch.cat([beta, beta[-1:].expand(self.Bp - self.B)])\n        if self.bitwise:\n            bwd = beta.reshape(self.W, WORD)\n            if not torch.all(bwd == bwd[:, :1]):\n                raise ValueError("bitwise multi-spin coding needs one temperature per word of 32 replicas")\n            # in place, like beta: a captured CUDA graph reads the table by address\n            self.ptab.copy_(torch.as_tensor(self._table(self.sys, bwd[:, 0].double().cpu().numpy()), device=self.device))\n        # in place: a CUDA graph captured earlier reads this buffer by address, so a new tensor would be ignored\n        self.beta.copy_(beta)\n\n    def set_seed(self, seed: int, sweep: int = 0):\n        self.state[0] = int(seed)\n        self.state[1] = int(sweep)\n\n    def init_spins(self, kind="random", seed: int = 0) -> torch.Tensor:\n        s = self.sys\n        g = torch.Generator(device="cpu").manual_seed(int(seed))\n        shape = (self.B, s.n) + s.shape\n        if kind == "ordered":\n            x = torch.zeros(shape)\n            x[:, 0] = 1.0\n        elif s.n == 1:\n            x = torch.randint(0, 2, shape, generator=g).float() * 2 - 1\n        else:\n            x = torch.randn(shape, generator=g)\n            x = x / x.norm(dim=1, keepdim=True)\n        x = x.to(self.device) * self._mask_f\n        if self.packed:\n            return self.pack(x)\n        return x.to(self.spin_dtype).contiguous()\n\n    def pack(self, x: torch.Tensor) -> torch.Tensor:\n        """(B, 1, b, L3, L2, L1) spins of +-1 (0 on masked sites) -> (W, 1, b, L3, L2, L1) int32 words."""\n        bits = (x > 0).to(torch.int64)\n        if self.Bp > self.B:\n            bits = torch.cat([bits, bits.new_zeros((self.Bp - self.B,) + tuple(bits.shape[1:]))])\n        bits = bits.reshape((self.W, WORD) + tuple(bits.shape[1:]))\n        sh = (torch.arange(WORD, device=bits.device, dtype=torch.int64)).reshape((1, WORD) + (1,) * (bits.dim() - 2))\n        word = (bits << sh).sum(dim=1)\n        word = torch.where(word >= 2 ** 31, word - 2 ** 32, word)\n        return word.to(torch.int32).contiguous()\n\n    def unpack(self, S: torch.Tensor) -> torch.Tensor:\n        """Packed words -> (B, 1, b, L3, L2, L1) int8 spins of +-1, 0 on masked sites; identity when unpacked."""\n        if not self.packed or S.dtype != torch.int32:     # already spins\n            return S\n        w = S.to(torch.int64) & 0xFFFFFFFF\n        sh = torch.arange(WORD, device=S.device, dtype=torch.int64).reshape((1, WORD) + (1,) * (S.dim() - 1))\n        bits = (w.unsqueeze(1) >> sh) & 1\n        x = (bits * 2 - 1).reshape((self.Bp,) + tuple(S.shape[1:]))[: self.B]\n        return (x * self._mask_f.to(torch.int64)).to(torch.int8)\n\n    # ------------------------------------------------------------ sweeps\n    def _launch(self, S, dst, c, a, grid_rows, row0=0, x_only=-1, sync=False, compact=False):\n        if self.bitwise:\n            if compact and self.rows_per > 1:\n                grid = (self.ngroups, -(-grid_rows // self.rows_per), 1)\n                kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n            elif compact:\n                grid = (self.ngroups, grid_rows, self.pchunks)\n                kw = dict(BLOCK=self.pblock, STRIDE=self.stride, BX=self.pblock, R=1, num_warps=self.nw)\n            else:\n                grid = (self.ngroups, grid_rows, self.pchunks_ref)\n                kw = dict(BLOCK=self.pblock_ref, STRIDE=1, BX=self.pblock_ref, R=1, num_warps=self.nw)\n            args = (S, dst, self.mask, self.k1, self.ptab, self.state, x_only, row0, self.W)\n            kw["G"] = self.G\n            if self.device.type == "cuda":\n                with torch.cuda.device(self.device):\n                    self.kernel[grid](*args, ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n                return\n            self.kernel[grid](*args, ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n            return\n        if self.packed:\n            if compact and self.rows_per > 1:\n                grid = (self.W, -(-grid_rows // self.rows_per), 1)\n                kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n            elif compact:\n                grid = (self.W, grid_rows, self.pchunks)\n                kw = dict(BLOCK=self.pblock, STRIDE=self.stride, BX=self.pblock, R=1, num_warps=4)\n            else:\n                grid = (self.W, grid_rows, self.pchunks_ref)\n                kw = dict(BLOCK=self.pblock_ref, STRIDE=1, BX=self.pblock_ref, R=1, num_warps=4)\n        elif compact:\n            grid = (self.B, -(-grid_rows // self.rows_per), self.cchunks)\n            kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n        else:\n            grid = (self.B, grid_rows, self.chunks)\n            kw = dict(BLOCK=self.block, STRIDE=1, BX=self.block, R=1, num_warps=self.nw)\n        if self.device.type == "cuda":\n            with torch.cuda.device(self.device):\n                self.kernel[grid](S, dst, self.mask, self.k1, self.beta, self.state, x_only, row0,\n                                  ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n            return\n        self.kernel[grid](S, dst, self.mask, self.k1, self.beta, self.state, x_only, row0,\n                          ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n\n    def sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """n coloured sweeps in place; advances the sweep counter."""\n        for _ in range(n):\n            for c, a in self.classes:\n                self._launch(S, S, c, a, self.nrows, compact=self.stride > 1)\n            self.state[1] += 1\n        return S\n\n    def graph(self, S: torch.Tensor, k: int):\n        """Capture k sweeps (launches and counter increments) in a CUDA graph; returns its replay.\n\n        Removes the host launch cost from timings; the counter stream is the same as k calls of\n        `sweep`, because the sweep counter is incremented on the device inside the graph.\n        """\n        with torch.cuda.device(self.device):\n            return self._graph(S, k)\n\n    def _graph(self, S: torch.Tensor, k: int):\n        s = torch.cuda.Stream(device=self.device)\n        s.wait_stream(torch.cuda.current_stream(self.device))\n        with torch.cuda.stream(s):\n            self.sweep(S, 1)          # warm-up compiles every class launch outside the capture\n        torch.cuda.current_stream(self.device).wait_stream(s)\n        torch.cuda.synchronize(self.device)\n        g = torch.cuda.CUDAGraph()\n        with torch.cuda.graph(g):\n            self.sweep(S, k)\n        dev = self.device\n\n        def replay():\n            with torch.cuda.device(dev):\n                g.replay()\n        # the graph addresses this sampler\'s buffers and S directly: the replay keeps them alive, or a caller\n        # that drops the sampler (or rebinds S) before replaying reads freed memory\n        replay.keep = (self, S, g)\n        return replay\n\n    def sync_sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """The negative control of E2: every site updated in the same step from the old state."""\n        for _ in range(n):\n            dst = S.clone()\n            for a in range(self.sys.b):\n                self._launch(S, dst, -1, a, self.nrows, sync=True)\n            S.copy_(dst)\n            self.state[1] += 1\n        return S\n\n    def reference_sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """Sequential reference chain: one launch per site, same kernel, same counters."""\n        L2, L1 = self.sys.L2, self.sys.L1\n        order = []\n        for c, a in self.classes:\n            sites = np.argwhere((self.colours[a] == c) & self.mask_np[a])   # (z, y, x), raster order\n            order.append((c, a, sites))\n        for _ in range(n):\n            for c, a, sites in order:\n                for z, y, x in sites:\n                    self._launch(S, S, c, a, 1, row0=int(z) * L2 + int(y), x_only=int(x))\n            self.state[1] += 1\n        return S\n\n    # ------------------------------------------------------------ observables\n    def local_field(self, S: torch.Tensor) -> torch.Tensor:\n        """Bond part of the local field (3) for every site, by rolls (float64), as a reference."""\n        s = self.sys\n        D = s.crystal.D\n        S = self.unpack(S)\n        S64 = S.double()\n        v = torch.zeros_like(S64)\n        dims = {0: 5, 1: 4, 2: 3}       # lattice axis k -> tensor dim\n        for i, bd in enumerate(s.crystal.bonds):\n            M = torch.as_tensor(s.bond_matrix(i), dtype=torch.float64, device=S.device)\n            if not M.any():\n                continue\n            src = S64[:, :, bd.alpha2]\n            valid = torch.ones(s.shape[1:], dtype=torch.float64, device=S.device)\n            for k in range(D):\n                d = bd.delta[k]\n                if d == 0:\n                    continue\n                src = torch.roll(src, shifts=-d, dims=dims[k] - 1)\n                if not s.periodic[k]:\n                    Lk = s.box[k]\n                    idx = torch.arange(Lk, device=S.device)\n                    ok = ((idx + d) >= 0) & ((idx + d) < Lk)\n                    shp = [1, 1, 1]\n                    shp[2 - k] = Lk\n                    valid = valid * ok.to(torch.float64).reshape(shp)\n            src = src * valid\n            v[:, :, bd.alpha] += torch.einsum("kl,blzyx->bkzyx", M, src)\n        return v\n\n    def energy(self, S: torch.Tensor) -> torch.Tensor:\n        """Total energy per replica, eq. (1), float64."""\n        s = self.sys\n        S = self.unpack(S)\n        S64 = S.double()\n        v = self.local_field(S)\n        E = -0.5 * (S64 * v).sum(dim=(1, 2, 3, 4, 5))\n        if s.h is not None:\n            h = torch.as_tensor(s.h, dtype=torch.float64, device=S.device).reshape(1, -1, 1, 1, 1, 1)\n            E = E - (S64 * h).sum(dim=(1, 2, 3, 4, 5))\n        if s.cubic_k1 is not None:\n            x, y, z = S64[:, 0], S64[:, 1], S64[:, 2]\n            k1 = self.k1.double()\n            a = k1 * ((x * y) ** 2 + (x * z) ** 2 + (y * z) ** 2) + s.cubic_k2 * (x * y * z) ** 2\n            E = E + a.sum(dim=(1, 2, 3, 4))\n        return E\n\n    def magnetization(self, S: torch.Tensor) -> torch.Tensor:\n        """Per-sublattice magnetization M_alpha, shape (B, b, n), float64."""\n        return self.unpack(S).double().sum(dim=(3, 4, 5)).transpose(1, 2)\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "86a9893be5ac75b74ca1e1a75acf15aadb336e4c5f1514453f6e32977fd3c52b", "sampler.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/sampler.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/sampler.py  sha256 86a9893be5ac75b7…")

In [ ]:
# Spin_Colour_Kernel/anneal.py — byte-identical to the repository. sha256 96da6965a875859f2b368a25ba83b7d28db8c4c2e6070c84773dd6d6075448f9
import hashlib, os
_src = '"""Cooling ladder of the magnetic nanodisk on the generated kernel.\n\nOne state is one Hamiltonian (seven parameters), one compiled kernel and one batch of replicas. Every\nreplica follows the same ladder from T_start to T_end (kelvin); at each temperature it thermalizes\nn_therm sweeps and measures every `every` sweeps during n_meas sweeps. The accumulators hold\nfluctuations about a reference taken once per temperature, so float cancellation cannot destroy\nC_v and chi, and `reduce_obs` gives the metrics from\n\n    obs[t] = (<dE>, <dE^2>, <dM>, <dM^2>, <Mz>, <Q>, E_ref, M_ref)      shape (n_T, 8, B)\n\nwith E the total energy (units of Jex), M = |sum_i s_i| / N_active, Mz its z component, and Q the\nBerg-Luscher charge of the middle layer. Design choices:\n- the update is the generated multi-colour kernel (any of the sc shells 1-4, colouring searched and\n  verified), launched from CUDA graphs of `every` sweeps;\n- observables are read every `every` sweeps, not after every sweep (they cost more than a sweep);\n- energies and magnetizations are accumulated in float64.\n\nRun as a module to process several states on one GPU and write one .npz per state:\n    python -m Spin_Colour_Kernel.anneal --config run.json --states 0,2 --out DIR\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom . import system as sy\nfrom .sampler import Sampler\n\nKB = 0.086173404     # meV / K (Jex = 1 sets the energy unit)\nPARAM_NAMES = ("Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM")\n\n\ndef build_state(params, Rd: float, L: int, angle_deg: float, shells=(1,)):\n    """The nanodisk System for one parameter row [Jex2, Jex3, Jex4, Kan1, KanS, Hex, KDM]."""\n    J2, J3, J4, K1, KS, H, D = (float(x) for x in params)\n    return sy.nanodot(Rd=Rd, L=L, Jex=1.0, Kan1=K1, Kan2=0.0, KanS=KS, Hex=H, KDM=D, gamma_deg=angle_deg,\n                      shells=tuple(shells), J_shells={2: J2, 3: J3, 4: J4})\n\n\ndef active_shells(rows) -> list[int]:\n    """Union over states of the shells with a non-zero coupling: one colouring serves every state."""\n    out = {1}\n    for r in rows:\n        for k, v in zip((2, 3, 4), r[:3]):\n            if float(v) != 0.0:\n                out.add(k)\n    return sorted(out)\n\n\ndef check_exact(s, device, sweeps: int = 3, T: float = 0.5, seed: int = 1) -> int:\n    """Differing spins between the coloured kernel and the sequential reference chain (must be 0)."""\n    A, R = Sampler(s, 2, device, seed), Sampler(s, 2, device, seed)\n    A.set_temperature(T)\n    R.set_temperature(T)\n    S1 = A.init_spins(seed=seed)\n    S2 = S1.clone()\n    A.sweep(S1, sweeps)\n    R.reference_sweep(S2, sweeps)\n    return int((S1 != S2).sum())\n\n\ndef _topo(layer: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:\n    """Berg-Luscher charge of one layer; layer (B, 3, N, N) indexed [y, x], valid (N-1, N-1)."""\n    s = layer.permute(0, 2, 3, 1)\n    s00, s10, s01, s11 = s[:, :-1, :-1], s[:, :-1, 1:], s[:, 1:, :-1], s[:, 1:, 1:]\n\n    def om(a, b, c):\n        num = (a * torch.cross(b, c, dim=-1)).sum(-1)\n        den = 1.0 + (a * b).sum(-1) + (b * c).sum(-1) + (c * a).sum(-1)\n        return 2.0 * torch.atan2(num, den)\n    w = om(s00, s10, s11) + om(s00, s11, s01)\n    return (w * valid).sum(dim=(1, 2)) / (4.0 * math.pi)\n\n\nclass Observer:\n    """E (float64), mean magnetization vector and middle-layer Q per replica, safe inside a CUDA graph.\n\n    Same quantities as Sampler.energy / Sampler.magnetization, but every constant (bond matrices, open-\n    boundary masks, field, anisotropy) is built once here: inside a graph capture no host-to-device copy\n    may run, and those methods build their constants on every call.\n    """\n\n    def __init__(self, A: Sampler):\n        s = A.sys\n        dev = A.k1.device\n        self.s, self.n_act = s, float(s.n_active)\n        self.bonds = []\n        dims = {0: 5, 1: 4, 2: 3}\n        for i, bd in enumerate(s.crystal.bonds):\n            M = s.bond_matrix(i)\n            if not M.any():\n                continue\n            valid = torch.ones(s.shape[1:], dtype=torch.float64, device=dev)\n            shifts = []\n            for k in range(s.crystal.D):\n                d = bd.delta[k]\n                if d == 0:\n                    continue\n                shifts.append((-d, dims[k] - 1))\n                if not s.periodic[k]:\n                    idx = torch.arange(s.box[k], device=dev)\n                    ok = ((idx + d) >= 0) & ((idx + d) < s.box[k])\n                    shp = [1, 1, 1]\n                    shp[2 - k] = s.box[k]\n                    valid = valid * ok.to(torch.float64).reshape(shp)\n            self.bonds.append((bd.alpha, bd.alpha2, shifts, valid,\n                               torch.as_tensor(M, dtype=torch.float64, device=dev)))\n        self.h = (None if s.h is None else\n                  torch.as_tensor(s.h, dtype=torch.float64, device=dev).reshape(1, -1, 1, 1, 1, 1))\n        self.k1 = None if s.cubic_k1 is None else A.k1.double()\n        self.k2 = float(s.cubic_k2)\n        self.mid = s.L3 // 2\n        disk = torch.as_tensor(s.site_mask()[0, self.mid], device=dev)\n        self.valid2d = (disk[:-1, :-1] & disk[:-1, 1:] & disk[1:, :-1] & disk[1:, 1:]).double()\n\n    def __call__(self, S: torch.Tensor):\n        S64 = S.double()\n        v = torch.zeros_like(S64)\n        for a, a2, shifts, valid, M in self.bonds:\n            src = S64[:, :, a2]\n            for sh, dim in shifts:\n                src = torch.roll(src, shifts=sh, dims=dim)\n            v[:, :, a] += torch.einsum("kl,blzyx->bkzyx", M, src * valid)\n        E = -0.5 * (S64 * v).sum(dim=(1, 2, 3, 4, 5))\n        if self.h is not None:\n            E = E - (S64 * self.h).sum(dim=(1, 2, 3, 4, 5))\n        if self.k1 is not None:\n            x, y, z = S64[:, 0], S64[:, 1], S64[:, 2]\n            E = E + (self.k1 * ((x * y) ** 2 + (x * z) ** 2 + (y * z) ** 2)\n                     + self.k2 * (x * y * z) ** 2).sum(dim=(1, 2, 3, 4))\n        Mv = S64.sum(dim=(2, 3, 4, 5)) / self.n_act\n        Q = _topo(S64[:, :, 0, self.mid], self.valid2d)\n        return E, Mv, Q\n\n\ndef anneal_state(s, replicas: int, temps_K, n_therm: int, n_meas: int, every: int, seed: int,\n                 device="cuda", log=print):\n    """Run the ladder for one state; returns obs (n_T, 8, B), mvec (n_T, B, 3), the final middle-layer\n    s_z of replica 0 per temperature (n_T, N, N), and timings."""\n    A = Sampler(s, replicas, device, seed)\n    S = A.init_spins(seed=seed)\n    dev = S.device\n    n_act = float(s.n_active)\n    observe = Observer(A)\n    k = max(1, int(every))\n    cuda = str(dev).startswith("cuda")\n    big = 10 * k\n    nT = len(temps_K)\n    obs = np.zeros((nT, 8, replicas))\n    mvec = np.zeros((nT, replicas, 3), np.float32)\n    snaps = np.zeros((nT,) + tuple(observe.valid2d.shape[i] + 1 for i in range(2)), np.float32)\n    f64 = dict(dtype=torch.float64, device=dev)\n    E_ref, M_ref = torch.zeros(replicas, **f64), torch.zeros(replicas, **f64)\n    acc = torch.zeros(6, replicas, **f64)                     # dE, dE^2, dM, dM^2, Mz, Q\n    accv = torch.zeros(replicas, 3, **f64)\n\n    def chunk():           # k sweeps, then one observation folded into the accumulators\n        A.sweep(S, k)\n        E, Mv, Q = observe(S)\n        dE = E - E_ref\n        dM = Mv.norm(dim=1) - M_ref\n        acc.add_(torch.stack([dE, dE * dE, dM, dM * dM, Mv[:, 2], Q]))\n        accv.add_(Mv)\n\n    A.set_temperature(KB * float(temps_K[0]))\n    if cuda:\n        # the thermalization block and the measure-and-accumulate step each become one CUDA graph: no\n        # host launches, no host-device copies inside the ladder (beta is written in place)\n        rep_big = A.graph(S, big)\n        side = torch.cuda.Stream(device=dev)\n        side.wait_stream(torch.cuda.current_stream(dev))\n        with torch.cuda.stream(side):\n            chunk()                                            # warm-up outside the capture\n        torch.cuda.current_stream(dev).wait_stream(side)\n        g = torch.cuda.CUDAGraph()\n        with torch.cuda.graph(g):\n            chunk()\n        rep_obs = g.replay\n    else:\n        rep_big, rep_obs = (lambda: A.sweep(S, big)), chunk\n\n    t0 = time.perf_counter()\n    for t, TK in enumerate(temps_K):\n        A.set_temperature(KB * float(TK))\n        for _ in range(n_therm // big):\n            rep_big()\n        if n_therm % big:\n            A.sweep(S, n_therm % big)\n        E, Mv, _ = observe(S)\n        E_ref.copy_(E)\n        M_ref.copy_(Mv.norm(dim=1))\n        acc.zero_()\n        accv.zero_()\n        m = max(1, n_meas // k)\n        for _ in range(m):\n            rep_obs()\n        a = (acc / m).cpu().numpy()\n        obs[t, :6] = a\n        obs[t, 6] = E_ref.cpu().numpy()\n        obs[t, 7] = M_ref.cpu().numpy()\n        mvec[t] = (accv / m).float().cpu().numpy()\n        snaps[t] = S[0, 2, 0, observe.mid].float().cpu().numpy()\n        if t % max(1, nT // 10) == 0 or t == nT - 1:\n            log(f"    T = {TK:7.3f} K   <E>/site = {(obs[t, 6].mean() + obs[t, 0].mean()) / n_act:+.4f}"\n                f"   |M| = {obs[t, 7].mean() + obs[t, 2].mean():.4f}   {time.perf_counter() - t0:7.1f} s")\n    if cuda:\n        torch.cuda.synchronize(dev)\n    dt = time.perf_counter() - t0\n    sweeps = nT * (n_therm + m * k)\n    return dict(obs=obs, mvec=mvec, snaps=snaps, seconds=dt, sweeps_per_chain=sweeps,\n                flips_per_ns=replicas * n_act * sweeps / dt / 1e9)\n\n\ndef reduce_obs(obs, temps, n_spins, kB: float = KB):\n    """Reduction: accumulators (n_T, 8, B) -> per-temperature scalars."""\n    obs = np.asarray(obs)\n    assert obs.ndim == 3 and obs.shape[1] == 8, f"expected (n_T, 8, batch), got {obs.shape}"\n    dE, dE2, dM, dM2, Mz, Q, E_ref, M_ref = [obs[:, i] for i in range(8)]\n    T = np.asarray(temps)[:, None]\n    var_E = dE2 - dE ** 2\n    var_M = dM2 - dM ** 2\n    E = dE + E_ref\n    M = dM + M_ref\n    Cv = var_E / (kB * T ** 2 * n_spins)\n    chi = var_M / (kB * T)\n    return dict(T=np.asarray(temps), E=E.mean(1) / n_spins, E_sd=E.std(1) / n_spins, M=M.mean(1),\n                M_sd=M.std(1), Mz=Mz.mean(1), Cv=Cv.mean(1), chi=chi.mean(1), Q=Q.mean(1), Q_sd=Q.std(1))\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", type=Path, required=True)\n    ap.add_argument("--states", required=True, help="comma list of state indices")\n    ap.add_argument("--out", type=Path, required=True)\n    ap.add_argument("--device", default="cuda:0")\n    a = ap.parse_args(argv)\n    cfg = json.loads(a.config.read_text())\n    a.out.mkdir(parents=True, exist_ok=True)\n    rows = cfg["params"]\n    shells = active_shells(rows)\n    temps = np.linspace(cfg["T_start"], cfg["T_end"], int(cfg["n_temps"]))\n    n_therm = int(cfg["sweeps"]) * 2 // 3\n    n_meas = int(cfg["sweeps"]) - n_therm\n    for i in (int(x) for x in a.states.split(",") if x):\n        name = cfg["names"][i]\n        s = build_state(rows[i], cfg["Rd"], cfg["layers"], cfg["angle"], shells)\n        d = check_exact(s, a.device)\n        print(f"[{name}] {s.colouring.n_colours} colours ({s.colouring.describe()}), "\n              f"exactness vs sequential chain: {d} differing spins", flush=True)\n        if d:\n            raise SystemExit(f"[{name}] the coloured kernel does not reproduce the sequential chain")\n        r = anneal_state(s, int(cfg["replicas"]), temps, n_therm, n_meas, int(cfg["every"]),\n                         int(cfg["seed"]) + i, a.device, log=lambda m: print(f"[{name}]" + m, flush=True))\n        np.savez_compressed(a.out / f"state_{name}.npz", params=np.asarray(rows[i], np.float32),\n                            obs=r["obs"], M=np.transpose(r["mvec"], (1, 0, 2)), snaps=r["snaps"],\n                            temperatures=temps.astype(np.float32), n_active=s.n_active,\n                            seconds=r["seconds"], flips_per_ns=r["flips_per_ns"],\n                            colours=s.colouring.n_colours)\n        print(f"[{name}] done in {r[\'seconds\']:.1f} s, {r[\'flips_per_ns\']:.2f} spin flips/ns", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "96da6965a875859f2b368a25ba83b7d28db8c4c2e6070c84773dd6d6075448f9", "anneal.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/anneal.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/anneal.py  sha256 96da6965a875859f…")

## 4 · Reading and checking the parameters

In [ ]:
# Read the parameters, count the states, and build one state to show the colouring the run will use.
import numpy as np
import importlib, sys
sys.path.insert(0, ".")
import Spin_Colour_Kernel.anneal as AN

P = np.asarray(PARAMS, dtype=float)
if P.ndim != 2 or P.shape[1] != 7:
    raise SystemExit(f"PARAMS must be a list of rows of 7 numbers; got shape {P.shape}")
N_STATES = P.shape[0]
names = (list(NAMES) + [f"state-{i+1}" for i in range(len(NAMES), N_STATES)])[:N_STATES]
if len(set(names)) != N_STATES:
    raise SystemExit("NAMES must be distinct: each state is written to its own file")

print(f"{N_STATES} state(s)")
print(f"{'':>3} {'state':<12}" + "".join(f"{n:>8}" for n in AN.PARAM_NAMES))
for i, nm in enumerate(names):
    print(f"{i+1:>3} {nm:<12}" + "".join(f"{v:>8.4f}" for v in P[i]))

shells = AN.active_shells(P)
probe = AN.build_state(P[0], RD, LAYERS, ANGLE, shells)
print(f"\nactive shells: {shells}")
print(f"colouring: {probe.colouring.describe()}  ->  {probe.colouring.n_colours} colours, "
      f"checked admissible on the disk")
print(f"disk: N = {probe.box[0]}, {LAYERS} layers, {probe.n_active} active sites")

n_therm = SWEEPS * 2 // 3
print(f"\nwork: {N_STATES} states x {REPLICAS} replicas, {N_TEMPS} temperatures x {SWEEPS} sweeps "
      f"({n_therm} thermalize, {SWEEPS - n_therm} measure, read every {MEASURE_EVERY})")
print(f"      {N_STATES * REPLICAS * N_TEMPS * SWEEPS * probe.n_active:.3e} spin updates in all")

## 5 · The run

In [ ]:
# Run: one process per GPU, states dealt round-robin. Each process first checks every state spin by
# spin against the sequential reference chain (3 sweeps, 2 replicas) and refuses to run a state that
# differs; then it runs the ladder and writes state_<name>.npz.
import json, os, threading, time

devs = list(range(torch.cuda.device_count())) if GPUS == "all" else list(GPUS)
devs = devs[:N_STATES] or [0]
cfg = dict(params=P.tolist(), names=names, Rd=RD, layers=LAYERS, angle=ANGLE, T_start=T_START,
           T_end=T_END, n_temps=N_TEMPS, sweeps=SWEEPS, replicas=REPLICAS, every=MEASURE_EVERY, seed=SEED)
with open("run.json", "w") as f:
    json.dump(cfg, f, indent=1)
os.makedirs("results", exist_ok=True)

procs = []
for j, g in enumerate(devs):
    mine = [i for i in range(N_STATES) if i % len(devs) == j]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONPATH=os.getcwd())
    cmd = [sys.executable, "-u", "-m", "Spin_Colour_Kernel.anneal", "--config", "run.json",
           "--states", ",".join(map(str, mine)), "--out", "results", "--device", "cuda:0"]
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    procs.append((g, mine, p))
    print(f"GPU {g}: states {[names[i] for i in mine]}")

def _pump(g, p):
    for line in p.stdout:
        print(f"GPU {g} {line}", end="", flush=True)

t0 = time.perf_counter()
th = [threading.Thread(target=_pump, args=(g, p)) for g, _, p in procs]
for t in th: t.start()
for t in th: t.join()
bad = [(g, p.wait()) for g, _, p in procs if p.wait() != 0]
print(f"\ntotal: {(time.perf_counter() - t0) / 60:.2f} min")
if bad:
    raise SystemExit(f"failed on GPU(s) {bad}: read the lines above")

## 6 · Physical metrics and the dataset

In [ ]:
# The physical metrics:
#
#   E/site  total energy over active sites, each bond counted once
#   |M|     norm of the mean magnetization over active sites; Mz its z component
#   Cv      <(ΔE)²> / (k_B T² N)       from fluctuations about a per-temperature reference (float64)
#   χ       <(Δ|M|)²> / (k_B T)
#   Q       Berg–Lüscher charge of the middle layer (quantised)
import json
out, transitions, Mvec = {}, {}, {}
for nm in names:
    z = np.load(f"results/state_{nm}.npz")
    red = AN.reduce_obs(z["obs"], z["temperatures"], int(z["n_active"]))
    out[nm] = {k: np.asarray(v).tolist() for k, v in red.items()}
    out[nm].update(seconds=float(z["seconds"]), flips_per_ns=float(z["flips_per_ns"]),
                   colours=int(z["colours"]), _snap=z["snaps"][-1].tolist())
    Mvec[nm] = z["M"]

print(f"{'state':<12}{'T':>7}{'E/site':>11}{'|M|':>9}{'Mz':>9}{'Cv':>10}{'χ':>10}{'Q':>9}"
      f"{'flips/ns':>10}")
for nm in names:
    r = out[nm]
    print(f"{nm:<12}{r['T'][-1]:>7.2f}{r['E'][-1]:>11.5f}{r['M'][-1]:>9.4f}{r['Mz'][-1]:>9.4f}"
          f"{r['Cv'][-1]:>10.4f}{r['chi'][-1]:>10.4f}{r['Q'][-1]:>9.4f}{r['flips_per_ns']:>10.2f}")

# The transition, located at the peak of each response function on the 0.1 K grid (no interpolation:
# a parabola through three noisy points claims a precision the data do not have). A finite disk shows
# a rounded maximum, not a divergence; a peak at either end of the ladder means the transition is
# outside the range and the number means nothing.
print(f"\n{'state':<12}{'T(Cv peak)':>12}{'Cv max':>10}{'T(χ peak)':>12}{'χ max':>10}  note")
for nm in names:
    r = out[nm]
    T = np.asarray(r["T"])
    row, msg = {}, []
    for k in ("Cv", "chi"):
        v = np.where(np.isfinite(r[k]), r[k], -np.inf)
        j = int(np.argmax(v))
        row[f"T_{k}"], row[f"{k}_max"] = float(T[j]), float(v[j])
        if j in (0, len(T) - 1):
            msg.append(f"{k} peak at the edge")
    transitions[nm] = row
    print(f"{nm:<12}{row['T_Cv']:>12.1f}{row['Cv_max']:>10.4f}{row['T_chi']:>12.1f}{row['chi_max']:>10.4f}"
          f"  {'; '.join(msg) or 'ok'}")

with open("results.json", "w") as f:
    json.dump(dict(params=P.tolist(), names=names, config=cfg, shells=shells,
                   colouring=probe.colouring.describe(), n_active=probe.n_active,
                   results=out, transitions=transitions), f, indent=1)
print("\nsaved results.json")
print("dataset: results/state_<name>.npz, one per state, each with")
print("  params (7,)  M (replicas, temperatures, 3)  temperatures (n_T,)  obs (n_T, 8, replicas)  snaps")
for nm in names:
    print(f"  results/state_{nm}.npz   M {Mvec[nm].shape} [replica, temperature, component]")

## 7 · Figures

In [ ]:
# Cooling curves per state, and the final middle-layer s_z of replica 0.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for nm in names:
    r = out[nm]
    for a, k, t in ((ax[0, 0], "E", "energy per site"), (ax[0, 1], "M", "|M|"),
                    (ax[1, 0], "Cv", "Cv"), (ax[1, 1], "chi", "χ")):
        a.plot(r["T"], r[k], lw=1.4, label=nm)
        a.set_xlabel("T (K)"); a.set_ylabel(t); a.grid(alpha=.25); a.invert_xaxis()
ax[0, 0].legend(fontsize=9)
fig.suptitle(f"{N_STATES} state(s) · {REPLICAS} replicas · {N_TEMPS}×{SWEEPS} sweeps", y=1.0)
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, N_STATES, figsize=(3.1 * N_STATES, 3.3))
axs = np.atleast_1d(axs)
for a, nm in zip(axs, names):
    im = a.imshow(np.asarray(out[nm]["_snap"]), cmap="jet", vmin=-1, vmax=1, origin="lower")
    a.set_title(f"{nm}  ($s_z$)", fontsize=10); a.axis("off")
fig.colorbar(im, ax=axs, fraction=.025)
plt.show()